# 03 Regression Analysis

This notebook estimates the realised-weather regression models used in the statistical analysis. It reads the standardised realised-weather panel from notebook 01 and produces descriptive checks, apparent-temperature and percentile features, category-level regression tables, store-day appendix regressions, VIF diagnostics, and selected figures.


## Pipeline position

This notebook follows `notebooks/02_basket_analysis.ipynb` and precedes `notebooks/04_synthetic_weather_model.ipynb`. It is the realised-weather statistical analysis step, not the operational ML forecasting step.


## Inputs

The primary modelling input is `data/processed/master_panel_realized_weather_windows.parquet`, filtered by `SELECTED_REALISED_WEATHER_WINDOW`. Reconstructed `*_obs_*` weather variables are mapped into the established regression feature names before apparent temperature and percentile features are recomputed.

Compatibility inputs `data/processed/regression_panel_realized_weather.parquet` and `data/processed/master_panel_realized_weather.parquet` may be loaded for benchmarking only. The original descriptive table is reproduced from `data/processed/descriptive_sales_weather_panel.parquet`. Weather-window diagnostics also use `data/Dataset.parquet`, `data/processed/realised_weather_daily_windows.parquet`, `data/processed/forecast_meps_daily_windows.parquet`, and `data/processed/forecast_meps_horizon_hour_audit.parquet`.


## Outputs

When executed, the notebook writes updated-methodology outputs under `outputs/regression_analysis/`: thesis-ready HTML tables, CSV inspection copies, optional diagnostic figures, and the weather-window specification table `Table_W_weather_window_specification_choice.*`. It also reproduces the original cell 17 descriptive statistics as `Hovedfilen_cell17_descriptive_statistics.*` and matching missing-value tables.

The active category-level regression outputs use open store-days only (`Closed == 0`, `Avdeling_total > 0`), included analysis categories, and genuine zero category-sales rows retained. Files in `thesis_context/tables/` are visual/style references only, not numerical targets.


## Methodological notes

The main regression unit is the store-category-day. The dependent variable is `log_Salg = log(1 + Antall_total)`, corresponding to `log(1 + Q_itk)`, because category sales may be zero on open store-days. The canonical category-level sample uses `Closed == 0`, `Avdeling_total > 0`, and `Analyse_Kategori` in `ANALYSIS_CATEGORIES`; it retains rows with `Antall_total == 0` and does not impose `Antall_total > 0`.

The analysis describes predictive and explanatory associations, not causal effects. Realised weather is valid in this regression section, but these realised-weather specifications must not be used later as operational ML forecast-weather inputs. Apparent temperature is used here as a realised thermal-comfort measure; later ML notebooks should use forecastable weather components instead of realised target-day apparent temperature.


## Table style and output mapping

The files in `thesis_context/tables/` provide HTML/CSS style references only. The updated regression values are generated from the current zero-retaining methodology and may differ from older positive-sales-only reference tables.

| Source or reference | Table/output role | Notebook 03 output | Status |
|---|---|---|---|
| `thesis_context/tables/` | Visual/style references: booktabs-like HTML, Times New Roman, title and note conventions, readable rounding, A4 compatibility | Applied to active outputs under `outputs/regression_analysis/html/` | Style reference only; not a numeric target |
| Hovedfilen cell 17 | Display-only descriptive statistics by `Analyse_Kategori` for `NettoKr`, `AntallArtikler`, and `AntallSalg`; printed missing-value check by category | `outputs/regression_analysis/html/Hovedfilen_cell17_descriptive_statistics.html`; `outputs/regression_analysis/csv/Hovedfilen_cell17_descriptive_statistics.csv`; missing-value check under matching `Hovedfilen_cell17_missing_values_by_category.*` files | Reproduced as portable HTML/CSV from `data/processed/descriptive_sales_weather_panel.parquet`; original was display/print-only |
| Hovedfilen cell 21 | Display-only category ladder for `Hot Drinks`; old source used positive category sales only | `outputs/regression_analysis/html/TableX_Hot_Drinks_regression_ladder.html`; `outputs/regression_analysis/csv/TableX_Hot_Drinks_regression_ladder.csv` | Updated-methodology version: zero category-sales rows retained |
| Hovedfilen cell 23 | Earlier store-day `TableX_booktabs1.html` without F-test rows | Not separately reproduced | Intentionally omitted because cell 24 overwrites the same filename with the fuller final version |
| Hovedfilen cell 24 | Store-day `TableX_booktabs1.html` | `outputs/regression_analysis/html/TableX_booktabs1.html`; panel CSV copies | Active store-day output; style follows reference tables |
| Hovedfilen cell 27 | VIF summary | `outputs/regression_analysis/html/Table_A1_VIF_summary.html`; panel CSV copies | Active VIF output; style follows reference tables |
| Hovedfilen cell 30 | Category summary and appendix table structures | `Table3_category_summary.html`; `TableA2_significance_structure.html`; `TableA3_iqr_effects.html` under `outputs/regression_analysis/html/` | Active updated-methodology category outputs; values are expected to differ from old positive-sales-only references |
| Hovedfilen cell 32 | Legacy/placeholder `TableX`/`TableAX`/`TableAY` variants | Disabled by default | Legacy positive-sales-only variants should not be written to the active thesis-ready folder |

Legacy/intensive-margin positive-sales-only outputs should only be written under `outputs/regression_analysis/legacy_original_reproduction/` and labelled as legacy comparisons.


## Environment and portable paths

The notebook is intended for the `csvi_env` kernel. Paths are discovered from the project root and are not hardcoded to a local machine path.


In [ ]:
import gc
import re
from html import escape
from pathlib import Path

import numpy as np
import pandas as pd
from patsy import dmatrices
from scipy import stats

# Matplotlib rendering is opt-in because it can crash this environment.
RENDER_REGRESSION_FIGURES = False
plt = None
if RENDER_REGRESSION_FIGURES:
    import matplotlib

    matplotlib.use('Agg', force=True)
    import matplotlib.pyplot as plt

MARKER_FILES = ('README_FOR_CODEX.md', 'AGENTS.md', 'pyproject.toml')


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if any((candidate / marker).exists() for marker in MARKER_FILES):
            return candidate
    raise FileNotFoundError(
        'Could not find the project root. Start Jupyter from inside the project folder '
        'or make sure README_FOR_CODEX.md, AGENTS.md, or pyproject.toml exists.'
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_REGRESSION_DIR = OUTPUT_DIR / 'regression_analysis'
OUTPUT_HTML_DIR = OUTPUT_REGRESSION_DIR / 'html'
OUTPUT_CSV_DIR = OUTPUT_REGRESSION_DIR / 'csv'
OUTPUT_FIGURE_DIR = OUTPUT_REGRESSION_DIR / 'figures'
OUTPUT_LEGACY_DIR = OUTPUT_REGRESSION_DIR / 'legacy_original_reproduction'
OUTPUT_LEGACY_HTML_DIR = OUTPUT_LEGACY_DIR / 'html'
OUTPUT_LEGACY_CSV_DIR = OUTPUT_LEGACY_DIR / 'csv'
WRITE_LEGACY_POSITIVE_SALES_OUTPUTS = False

WEATHER_WINDOW_ORDER = ['full_day_00_23', 'trade_08_19']
SELECTED_REALISED_WEATHER_WINDOW = 'trade_08_19'
if SELECTED_REALISED_WEATHER_WINDOW not in WEATHER_WINDOW_ORDER:
    raise ValueError(f'SELECTED_REALISED_WEATHER_WINDOW must be one of {WEATHER_WINDOW_ORDER}')


REGRESSION_PANEL_PATH = PROCESSED_DIR / 'regression_panel_realized_weather.parquet'
DATASET_PATH = DATA_DIR / 'Dataset.parquet'
MASTER_PANEL_PATH = PROCESSED_DIR / 'master_panel_realized_weather.parquet'
DESCRIPTIVE_PANEL_PATH = PROCESSED_DIR / 'descriptive_sales_weather_panel.parquet'
MASTER_PANEL_WINDOWS_PATH = PROCESSED_DIR / 'master_panel_realized_weather_windows.parquet'
REALISED_WINDOWS_PATH = PROCESSED_DIR / 'realised_weather_daily_windows.parquet'
FORECAST_WINDOWS_PATH = PROCESSED_DIR / 'forecast_meps_daily_windows.parquet'
FORECAST_HOUR_AUDIT_PATH = PROCESSED_DIR / 'forecast_meps_horizon_hour_audit.parquet'

CHECKS_PATH = OUTPUT_CSV_DIR / 'regression_panel_checks.csv'
ORIGINAL_DESCRIPTIVE_CSV_PATH = OUTPUT_CSV_DIR / 'Hovedfilen_cell17_descriptive_statistics.csv'
ORIGINAL_DESCRIPTIVE_HTML_PATH = OUTPUT_HTML_DIR / 'Hovedfilen_cell17_descriptive_statistics.html'
ORIGINAL_DESCRIPTIVE_MISSING_CSV_PATH = OUTPUT_CSV_DIR / 'Hovedfilen_cell17_missing_values_by_category.csv'
ORIGINAL_DESCRIPTIVE_MISSING_HTML_PATH = OUTPUT_HTML_DIR / 'Hovedfilen_cell17_missing_values_by_category.html'
DESCRIPTIVE_CSV_PATH = OUTPUT_CSV_DIR / 'diagnostic_descriptive_by_category.csv'
DESCRIPTIVE_HTML_PATH = OUTPUT_HTML_DIR / 'diagnostic_descriptive_by_category.html'
ZERO_RATE_PATH = OUTPUT_CSV_DIR / 'zero_category_sales_open_days.csv'
WEATHER_MISSINGNESS_PATH = OUTPUT_CSV_DIR / 'weather_missingness.csv'
DEPENDENT_SUMMARY_PATH = OUTPUT_CSV_DIR / 'dependent_variable_summary.csv'
WEATHER_CORR_CSV_PATH = OUTPUT_CSV_DIR / 'weather_correlation.csv'
WEATHER_CORR_FIG_PATH = OUTPUT_FIGURE_DIR / 'weather_correlation_heatmap.png'
LOG_DIST_FIG_PATH = OUTPUT_FIGURE_DIR / 'log_sales_distribution_by_category.png'

WEATHER_WINDOW_SALES_CSV_PATH = OUTPUT_CSV_DIR / 'Table_W1_sales_distribution_by_candidate_window.csv'
WEATHER_WINDOW_SALES_HTML_PATH = OUTPUT_HTML_DIR / 'Table_W1_sales_distribution_by_candidate_window.html'
WEATHER_WINDOW_COVERAGE_CSV_PATH = OUTPUT_CSV_DIR / 'Table_W2_h2_forecast_coverage_by_weather_window.csv'
WEATHER_WINDOW_COVERAGE_HTML_PATH = OUTPUT_HTML_DIR / 'Table_W2_h2_forecast_coverage_by_weather_window.html'
WEATHER_WINDOW_PRECIP_CSV_PATH = OUTPUT_CSV_DIR / 'Table_W3_h2_precipitation_forecast_diagnostics_by_window.csv'
WEATHER_WINDOW_PRECIP_HTML_PATH = OUTPUT_HTML_DIR / 'Table_W3_h2_precipitation_forecast_diagnostics_by_window.html'
WEATHER_WINDOW_REGRESSION_CSV_PATH = OUTPUT_CSV_DIR / 'Table_W4_weather_window_regression_comparison.csv'
WEATHER_WINDOW_REGRESSION_HTML_PATH = OUTPUT_HTML_DIR / 'Table_W4_weather_window_regression_comparison.html'
WEATHER_WINDOW_DECISION_CSV_PATH = OUTPUT_CSV_DIR / 'Table_W_weather_window_specification_choice.csv'
WEATHER_WINDOW_DECISION_HTML_PATH = OUTPUT_HTML_DIR / 'Table_W_weather_window_specification_choice.html'

CATEGORY_LADDER_CSV_PATH = OUTPUT_CSV_DIR / 'TableX_Hot_Drinks_regression_ladder.csv'
CATEGORY_LADDER_HTML_PATH = OUTPUT_HTML_DIR / 'TableX_Hot_Drinks_regression_ladder.html'

STORE_DAY_PANELA_CSV_PATH = OUTPUT_CSV_DIR / 'TableX_booktabs1_panelA.csv'
STORE_DAY_PANELB_CSV_PATH = OUTPUT_CSV_DIR / 'TableX_booktabs1_panelB.csv'
STORE_DAY_BOOKTABS_HTML_PATH = OUTPUT_HTML_DIR / 'TableX_booktabs1.html'

VIF_CONTROL_CSV_PATH = OUTPUT_CSV_DIR / 'vif_control_full.csv'
VIF_WEATHER_CSV_PATH = OUTPUT_CSV_DIR / 'vif_weather_full.csv'
VIF_PANELA_CSV_PATH = OUTPUT_CSV_DIR / 'Table_A1_VIF_summary_panelA.csv'
VIF_PANELB_CSV_PATH = OUTPUT_CSV_DIR / 'Table_A1_VIF_summary_panelB.csv'
VIF_SUMMARY_HTML_PATH = OUTPUT_HTML_DIR / 'Table_A1_VIF_summary.html'

CATEGORY_SUMMARY_CSV_PATH = OUTPUT_CSV_DIR / 'Table3_category_summary.csv'
CATEGORY_SUMMARY_HTML_PATH = OUTPUT_HTML_DIR / 'Table3_category_summary.html'
SIGNIFICANCE_PANELA_CSV_PATH = OUTPUT_CSV_DIR / 'TableA2_significance_structure_panelA.csv'
SIGNIFICANCE_PANELB_CSV_PATH = OUTPUT_CSV_DIR / 'TableA2_significance_structure_panelB.csv'
SIGNIFICANCE_HTML_PATH = OUTPUT_HTML_DIR / 'TableA2_significance_structure.html'
IQR_EFFECTS_CSV_PATH = OUTPUT_CSV_DIR / 'TableA3_iqr_effects.csv'
IQR_EFFECTS_HTML_PATH = OUTPUT_HTML_DIR / 'TableA3_iqr_effects.html'

LEGACY_CATEGORY_SUMMARY_CSV_PATH = OUTPUT_LEGACY_CSV_DIR / 'TableX_category_summary.csv'
LEGACY_CATEGORY_SUMMARY_HTML_PATH = OUTPUT_LEGACY_HTML_DIR / 'TableX_category_summary.html'
LEGACY_SIGNIFICANCE_PANELA_CSV_PATH = OUTPUT_LEGACY_CSV_DIR / 'TableAX_significance_structure_panelA.csv'
LEGACY_SIGNIFICANCE_PANELB_CSV_PATH = OUTPUT_LEGACY_CSV_DIR / 'TableAX_significance_structure_panelB.csv'
LEGACY_SIGNIFICANCE_HTML_PATH = OUTPUT_LEGACY_HTML_DIR / 'TableAX_significance_structure.html'
LEGACY_IQR_EFFECTS_CSV_PATH = OUTPUT_LEGACY_CSV_DIR / 'TableAY_iqr_effects.csv'
LEGACY_IQR_EFFECTS_HTML_PATH = OUTPUT_LEGACY_HTML_DIR / 'TableAY_iqr_effects.html'

for folder in [OUTPUT_REGRESSION_DIR, OUTPUT_HTML_DIR, OUTPUT_CSV_DIR, OUTPUT_FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
if WRITE_LEGACY_POSITIVE_SALES_OUTPUTS:
    for folder in [OUTPUT_LEGACY_DIR, OUTPUT_LEGACY_HTML_DIR, OUTPUT_LEGACY_CSV_DIR]:
        folder.mkdir(parents=True, exist_ok=True)


def project_relative(path):
    return path.relative_to(PROJECT_ROOT).as_posix()


def require_file(path, description):
    if not path.exists():
        raise FileNotFoundError(
            f'Missing {description}: {project_relative(path)}. '
            'Run notebook 01 or place the file under the documented project folder before running this notebook.'
        )


def require_columns(frame, required_columns, frame_name):
    missing = [column for column in required_columns if column not in frame.columns]
    if missing:
        available_columns = list(frame.columns)
        print(f'{frame_name} available columns ({len(available_columns)}): {available_columns}')
        raise KeyError(f'{frame_name} is missing required columns: {missing}')


def report_memory(frame, name):
    memory_gb = frame.memory_usage(deep=True).sum() / 1_000_000_000
    print(f'{name}: shape={frame.shape}, approx_memory={memory_gb:.3f} GB')
    return memory_gb


def star(p_value):
    if pd.isna(p_value):
        return ''
    if p_value < 0.001:
        return '***'
    if p_value < 0.01:
        return '**'
    if p_value < 0.05:
        return '*'
    return ''


def stars_category_summary(p_value):
    if pd.isna(p_value):
        return ''
    if p_value < 0.01:
        return '***'
    if p_value < 0.05:
        return '**'
    if p_value < 0.10:
        return '*'
    return ''


def format_number_for_table(value, kind='number', decimals=None):
    if isinstance(value, str):
        return value
    if pd.isna(value):
        return ''
    if isinstance(kind, int):
        decimals = kind
        kind = 'number'
    numeric_value = float(value)
    if kind == 'count':
        return f'{int(round(numeric_value)):,}'
    if kind == 'p':
        return '<0.001' if numeric_value < 0.001 else f'{numeric_value:.3f}'
    if kind == 'r2':
        return f'{numeric_value:.3f}'
    if kind == 'percent':
        return f'{numeric_value:.1f}'
    if decimals is None:
        decimals = 2
    return f'{numeric_value:.{decimals}f}'


def format_dataframe_for_display(frame, formats=None):
    formats = {} if formats is None else formats
    table = frame.copy()
    for column, kind in formats.items():
        if column in table.columns:
            table[column] = table[column].map(lambda value: format_number_for_table(value, kind=kind))
    table = table.where(pd.notna(table), '')
    return table


def diagnostic_table_css(orientation='landscape', font_size='9.5pt', table_layout='fixed', extra_css=''):
    page_size = 'A4 landscape' if orientation == 'landscape' else 'A4 portrait'
    page_width = '297mm' if orientation == 'landscape' else '210mm'
    margin = '12mm' if orientation == 'landscape' else '16mm'
    html_body_width = (
        f'html, body {{ width: {page_width}; margin: 0; padding: 0; }}'
        if orientation == 'landscape'
        else ''
    )
    return f'''
<style>
  @page {{ size: {page_size}; margin: {margin}; }}
  {html_body_width}

  body {{ font-family: "Times New Roman", Times, serif; font-size: {font_size}; margin: 0; padding: 0; }}
  .title {{ text-align: center; font-size: 12pt; font-weight: bold; margin: 0 0 5px 0; }}
  .subtitle {{ text-align: center; font-size: 10pt; margin: 0 0 6px 0; }}
  .note {{ margin-top: 5px; font-size: 9.5pt; text-align: left; }}
  .rule {{ border-top: 1px solid #000; margin: 6px 0 0 0; }}

  table {{
    border-collapse: collapse;
    width: 100%;
    table-layout: {table_layout};
    border-top: 1.2px solid #000;
    border-bottom: 1.2px solid #000;
    margin: 0;
  }}

  th, td {{ border: none; padding: 2px 4px; vertical-align: top; }}
  thead th {{ text-align: center; font-weight: bold; padding-bottom: 4px; line-height: 1.05; }}
  thead tr:last-child th {{ border-bottom: 1px solid #000; }}
  tbody th {{ text-align: left; font-weight: normal; }}
  tbody td {{ text-align: center; }}
  tbody td:first-child {{ text-align: left; }}

  .tbl-main th:first-child, .tbl-main td:first-child {{ text-align: left; white-space: normal; }}
  .tbl-main th:not(:first-child), .tbl-main td:not(:first-child) {{ text-align: center; white-space: nowrap; }}

  @media print {{
    thead {{ display: table-row-group; }}
    tfoot {{ display: table-row-group; }}
  }}
</style>
<style>
{extra_css}
</style>
'''


def save_html_table(
    frame,
    html_path,
    title,
    notes,
    subtitle=None,
    orientation='landscape',
    table_class='tbl-main',
    index=False,
    extra_css='',
    font_size='9.5pt',
):
    table_frame = frame.copy().where(pd.notna(frame), '')
    table_html = table_frame.to_html(index=index, classes=table_class, border=0, escape=True, na_rep='')
    subtitle_html = f'<div class="subtitle">{escape(subtitle)}</div>' if subtitle else ''
    html = (
        '<html><head>'
        + diagnostic_table_css(orientation=orientation, font_size=font_size, extra_css=extra_css)
        + '</head><body><div class="page">'
        + f'<div class="title">{escape(title)}</div>'
        + subtitle_html
        + table_html
        + f'<div class="note">{escape(notes)}</div>'
        + '</div></body></html>'
    )
    html_path.write_text(html, encoding='utf-8')
    print(f'Saved: {project_relative(html_path)}')


def save_table_outputs(
    frame,
    csv_path,
    html_path,
    title,
    note,
    orientation='landscape',
    table_class='tbl-main',
    index=False,
    extra_css='',
    subtitle=None,
):
    output_frame = frame.copy().where(pd.notna(frame), '')
    output_frame.to_csv(csv_path, index=index)
    print(f'Saved: {project_relative(csv_path)}')
    save_html_table(
        output_frame,
        html_path,
        title=title,
        notes=note,
        subtitle=subtitle,
        orientation=orientation,
        table_class=table_class,
        index=index,
        extra_css=extra_css,
    )


def format_regression_cell(result, term, digits=4):
    if term not in result.params.index:
        return ''
    coef = float(result.params[term])
    t_value = float(result.tvalues[term])
    p_value = float(result.pvalues[term])
    return f'{coef:.{digits}f}{star(p_value)} ({t_value:.2f})'


def html_table_no_index(frame, class_name):
    return frame.to_html(index=False, escape=False, classes=class_name, border=0)


def hovedfilen_common_css(page_size='A4 landscape', margin='12mm', body_width=None):
    width_css = ''
    if body_width is not None:
        width_css = f'html, body {{ width: {body_width}; margin: 0; padding: 0; }}'
    return f'''
<style>
  @page {{ size: {page_size}; margin: {margin}; }}
  {width_css}

  body {{ font-family: "Times New Roman", Times, serif; font-size: 9.5pt; }}
  .title {{ text-align: center; font-size: 12pt; font-weight: bold; margin: 0 0 5px 0; }}
  .subtitle {{ text-align: left; font-size: 10pt; font-weight: bold; margin: 8px 0 4px 0; }}
  .rule  {{ border-top: 1px solid #000; margin: 6px 0 0 0; }}
  .note  {{ margin-top: 5px; font-size: 9.5pt; text-align: left; }}

  table {{
    border-collapse: collapse;
    width: 100%;
    table-layout: fixed;
    border-top: 1.2px solid #000;
    border-bottom: 1.2px solid #000;
    margin: 0;
  }}

  th, td {{ border: none; padding: 1px 3px; vertical-align: top; }}
  thead th {{ text-align: center; font-weight: bold; padding-bottom: 3px; line-height: 1.05; }}
  thead tr:last-child th {{ border-bottom: 1px solid #000; }}

  @media print {{
    thead {{ display: table-row-group; }}
    tfoot {{ display: table-row-group; }}
  }}
</style>
'''


print(f'Project root: {PROJECT_ROOT}')
print(f'Regression HTML folder: {project_relative(OUTPUT_HTML_DIR)}')
print(f'Regression CSV folder: {project_relative(OUTPUT_CSV_DIR)}')


## Input checks

The next cell verifies file existence and Parquet schema metadata without loading the full regression panel.


In [ ]:
REQUIRED_PANEL_COLUMNS = [
    'DatoSolgt',
    'AvdelingID',
    'AvdelingTekst',
    'Region',
    'Analyse_Kategori',
    'Ukedag',
    'Årstid',
    'Helligdager',
    'Antall_total',
    'Antall_campaign',
    'Antall_ordinary',
    'CampaignDay',
    'log_Salg',
    'Avdeling_total',
    'Closed',
    'precip_val',
    'temp_mean',
    'humidity_mean',
    'wind_mean',
    'cloud_mean',
]


WINDOW_WEATHER_COLUMN_MAP = {
    'temp_obs_mean': 'temp_mean',
    'precip_obs_sum': 'precip_val',
    'humid_obs_mean': 'humidity_mean',
    'wind_obs_mean': 'wind_mean',
    'cloud_obs_mean': 'cloud_mean',
}

REQUIRED_WINDOW_PANEL_COLUMNS = [
    'DatoSolgt',
    'AvdelingID',
    'AvdelingTekst',
    'Region',
    'Analyse_Kategori',
    'Ukedag',
    'Årstid',
    'Helligdager',
    'Antall_total',
    'Antall_campaign',
    'Antall_ordinary',
    'CampaignDay',
    'log_Salg',
    'Avdeling_total',
    'Closed',
    'aggregation_window',
    *WINDOW_WEATHER_COLUMN_MAP.keys(),
]


DESCRIPTIVE_PANEL_COLUMNS = [
    'DatoSolgt',
    'AvdelingID',
    'AvdelingTekst',
    'Region',
    'Analyse_Kategori',
    'NettoKr',
    'AntallArtikler',
    'AntallSalg',
    'Antall_total',
    'Avdeling_total',
    'Closed',
    'temp_mean',
    'precip_val',
    'humidity_mean',
    'wind_mean',
    'cloud_mean',
]


def parquet_column_status(path, required_columns):
    try:
        import pyarrow.parquet as pq

        parquet_file = pq.ParquetFile(path)
        available_columns = set(parquet_file.schema_arrow.names)
        metadata = {
            'file': project_relative(path),
            'exists': path.exists(),
            'size_mb': round(path.stat().st_size / 1_000_000, 2),
            'row_groups': parquet_file.num_row_groups,
            'metadata_rows': parquet_file.metadata.num_rows,
        }
        columns = pd.DataFrame(
            {
                'column': required_columns,
                'available': [column in available_columns for column in required_columns],
            }
        )
        return metadata, columns
    except Exception as exc:
        metadata = {
            'file': project_relative(path),
            'exists': path.exists(),
            'size_mb': round(path.stat().st_size / 1_000_000, 2) if path.exists() else np.nan,
            'row_groups': pd.NA,
            'metadata_rows': pd.NA,
            'note': f'Schema inspection failed: {exc}',
        }
        columns = pd.DataFrame({'column': required_columns, 'available': pd.NA})
        return metadata, columns


PANEL_PATH = MASTER_PANEL_WINDOWS_PATH
panel_input_note = f'active reconstructed realised-weather window: {SELECTED_REALISED_WEATHER_WINDOW}'
LEGACY_PANEL_PATH = REGRESSION_PANEL_PATH if REGRESSION_PANEL_PATH.exists() else MASTER_PANEL_PATH

require_file(MASTER_PANEL_WINDOWS_PATH, 'window-aware master panel from notebook 01')
panel_metadata, panel_column_status = parquet_column_status(PANEL_PATH, REQUIRED_WINDOW_PANEL_COLUMNS)
panel_metadata['input_note'] = panel_input_note
panel_metadata['selected_window'] = SELECTED_REALISED_WEATHER_WINDOW

metadata_rows = [panel_metadata]
if LEGACY_PANEL_PATH.exists():
    legacy_metadata, legacy_column_status = parquet_column_status(LEGACY_PANEL_PATH, REQUIRED_PANEL_COLUMNS)
    legacy_metadata['input_note'] = 'optional compatibility benchmark; not active main-regression source'
    legacy_metadata['selected_window'] = pd.NA
    metadata_rows.append(legacy_metadata)

display(pd.DataFrame(metadata_rows))
display(panel_column_status)


## Reconstructed realised-weather regression panel

The active regression panel is loaded from `master_panel_realized_weather_windows.parquet` and filtered to `SELECTED_REALISED_WEATHER_WINDOW`. Reconstructed realised-weather columns from notebook 01a are mapped to `temp_mean`, `precip_val`, `humidity_mean`, `wind_mean`, and `cloud_mean` so the established formulas and table structure remain comparable. Legacy full-day panels may be loaded for compatibility checks but are not the active source for the main regressions.


In [ ]:
def unique_preserving_order(values):
    return list(dict.fromkeys(values))


ACTIVE_WINDOW_PANEL_COLUMNS = unique_preserving_order(REQUIRED_WINDOW_PANEL_COLUMNS)


def read_window_filtered_parquet(path, selected_window, columns, frame_name):
    try:
        frame = pd.read_parquet(
            path,
            columns=columns,
            filters=[('aggregation_window', '==', selected_window)],
        )
    except Exception as exc:
        print(
            f'Filtered parquet read failed for {frame_name} ({exc}). '
            'Falling back to column-pruned read before filtering.'
        )
        frame_all = pd.read_parquet(path, columns=columns)
        frame = frame_all.loc[frame_all['aggregation_window'].eq(selected_window)].copy()
        del frame_all
        gc.collect()
    return frame


def build_regression_panel_from_window(window_panel, selected_window):
    require_columns(window_panel, REQUIRED_WINDOW_PANEL_COLUMNS, 'selected_window_panel')
    if window_panel.empty:
        raise ValueError(
            f'SELECTED_REALISED_WEATHER_WINDOW={selected_window!r} produced no rows. '
            'Check the available aggregation_window values in the source parquet.'
        )
    available_windows = sorted(window_panel['aggregation_window'].dropna().unique())
    if available_windows != [selected_window]:
        raise ValueError(
            f'Expected only {selected_window!r} after filtered read, found {available_windows}.'
        )

    d = window_panel.copy()
    for source, target in WINDOW_WEATHER_COLUMN_MAP.items():
        d[target] = pd.to_numeric(d[source], errors='coerce')

    d['DatoSolgt'] = pd.to_datetime(d['DatoSolgt']).dt.normalize()
    d['AvdelingID'] = pd.to_numeric(d['AvdelingID'], errors='raise').astype('int64')
    d['Antall_total'] = pd.to_numeric(d['Antall_total'], errors='coerce')
    d['log_Salg'] = np.log1p(d['Antall_total'])
    d['Closed'] = pd.to_numeric(d['Closed'], errors='coerce').fillna(0).astype('int8')

    duplicate_keys = int(d.duplicated(['DatoSolgt', 'AvdelingID', 'Analyse_Kategori']).sum())
    if duplicate_keys:
        raise ValueError(
            f'Active regression panel has {duplicate_keys:,} duplicate date-store-category '
            'keys after window filtering.'
        )
    return d


selected_window_panel = read_window_filtered_parquet(
    MASTER_PANEL_WINDOWS_PATH,
    SELECTED_REALISED_WEATHER_WINDOW,
    ACTIVE_WINDOW_PANEL_COLUMNS,
    'selected_window_panel',
)
regression_panel_window = build_regression_panel_from_window(selected_window_panel, SELECTED_REALISED_WEATHER_WINDOW)
del selected_window_panel
gc.collect()

regression_panel_active = regression_panel_window
regression_panel = regression_panel_active
regression_panel_legacy_available = LEGACY_PANEL_PATH.exists()

mapped_weather_missingness = (
    regression_panel_active[list(WINDOW_WEATHER_COLUMN_MAP.values())]
    .isna()
    .mean()
    .reset_index()
)
mapped_weather_missingness.columns = ['mapped_weather_column', 'missing_rate']

report_memory(regression_panel_active, 'regression_panel_active')
print(f'Loaded active reconstructed realised-weather panel: {project_relative(MASTER_PANEL_WINDOWS_PATH)}')
print(f'Selected realised-weather window: {SELECTED_REALISED_WEATHER_WINDOW}')
print(f'Rows in active panel: {len(regression_panel_active):,}')
print(f'Optional legacy panel available but not loaded: {regression_panel_legacy_available}')
display(mapped_weather_missingness)


## Panel checks

These checks summarize panel shape, open-day logic, zero category-sales rates on open store-days, missing weather, and the dependent-variable distribution.


In [ ]:
ANALYSIS_CATEGORIES = ['Dinner', 'Lunch', 'Sweets', 'Salads', 'Dessert', 'Cold Drinks', 'Hot Drinks']
WEATHER_COLUMNS = ['precip_val', 'temp_mean', 'humidity_mean', 'wind_mean', 'cloud_mean']

open_store_day_mask = regression_panel['Closed'].eq(0)
category_mask = regression_panel['Analyse_Kategori'].isin(ANALYSIS_CATEGORIES)
open_category_mask = open_store_day_mask & category_mask

zero_rate_total = float(regression_panel.loc[open_category_mask, 'Antall_total'].eq(0).mean())
zero_by_category = (
    regression_panel.loc[open_category_mask]
    .groupby('Analyse_Kategori', observed=True)['Antall_total']
    .apply(lambda values: float(values.eq(0).mean()))
    .reset_index()
)
zero_by_category.columns = ['Analyse_Kategori', 'zero_category_sales_rate_open_store_days']
zero_by_category.to_csv(ZERO_RATE_PATH, index=False)

weather_missingness = regression_panel[WEATHER_COLUMNS].isna().mean().reset_index()
weather_missingness.columns = ['weather_column', 'missing_rate']
weather_missingness.to_csv(WEATHER_MISSINGNESS_PATH, index=False)

dependent_summary = (
    regression_panel.loc[open_category_mask, ['Antall_total', 'log_Salg']]
    .describe()
    .transpose()
    .reset_index()
)
dependent_summary.columns = ['variable'] + dependent_summary.columns.tolist()[1:]
dependent_summary.to_csv(DEPENDENT_SUMMARY_PATH, index=False)

check_rows = [
    {'check': 'input_file', 'value': project_relative(PANEL_PATH), 'note': panel_input_note},
    {'check': 'row_count', 'value': len(regression_panel), 'note': 'Full realised-weather panel rows'},
    {'check': 'start_date', 'value': str(regression_panel['DatoSolgt'].min().date()), 'note': 'Minimum DatoSolgt'},
    {'check': 'end_date', 'value': str(regression_panel['DatoSolgt'].max().date()), 'note': 'Maximum DatoSolgt'},
    {'check': 'n_stores', 'value': regression_panel['AvdelingID'].nunique(), 'note': 'Unique stores'},
    {
        'check': 'n_categories',
        'value': regression_panel['Analyse_Kategori'].nunique(),
        'note': 'Unique categories including Ekskluderes if present',
    },
    {
        'check': 'open_store_category_rows',
        'value': int(open_category_mask.sum()),
        'note': 'Open store-days and included analysis categories',
    },
    {
        'check': 'zero_category_sales_rate_open_store_days',
        'value': zero_rate_total,
        'note': 'Open rows with Antall_total == 0; these are retained in category regressions',
    },
    {
        'check': 'closed_row_rate',
        'value': float(regression_panel['Closed'].mean()),
        'note': 'Row-level closed indicator mean',
    },
]
for column in WEATHER_COLUMNS:
    check_rows.append(
        {
            'check': f'missing_rate_{column}',
            'value': float(regression_panel[column].isna().mean()),
            'note': f'Missing rate for {column}',
        }
    )

checks_df = pd.DataFrame(check_rows)
checks_df.to_csv(CHECKS_PATH, index=False)

display(checks_df)
display(zero_by_category)
display(dependent_summary)

## Apparent temperature and weather percentiles

This step recomputes apparent temperature and weather-percentile features after selected-window realised weather has been mapped into the established regression column names. The active inputs are selected-window values from `temp_obs_mean`, `precip_obs_sum`, `humid_obs_mean`, `wind_obs_mean`, and `cloud_obs_mean`, not stale compatibility-weather values from the old full-day panel.


In [ ]:
def add_apparent_temperature(frame):
    temperature_c = frame['temp_mean']
    wind_ms = frame['wind_mean']
    relative_humidity = frame['humidity_mean']
    vapor_pressure = (relative_humidity / 100) * 6.105 * np.exp(17.27 * temperature_c / (237.7 + temperature_c))
    frame['apparent_temp'] = temperature_c + 0.33 * vapor_pressure - 0.70 * wind_ms - 4.00
    return frame


def add_weather_percentiles_consistent(frame):
    group_columns = ['Årstid', 'AvdelingID']
    frame['Temp_pct'] = frame.groupby(group_columns, observed=True)['temp_mean'].rank(pct=True)
    frame['Temp_pct_sq'] = frame['Temp_pct'] ** 2
    frame['AT_percentile'] = frame.groupby(group_columns, observed=True)['apparent_temp'].rank(pct=True)
    frame['AT_percentile_sq'] = frame['AT_percentile'] ** 2
    frame['Precip_wet_pct'] = frame.groupby(group_columns, observed=True)['precip_val'].rank(pct=True)
    frame['Precip_pct'] = 1 - frame['Precip_wet_pct']
    frame['Precip_pct_sq'] = frame['Precip_pct'] ** 2
    frame['Cloud_cloudy_pct'] = frame.groupby(group_columns, observed=True)['cloud_mean'].rank(pct=True)
    frame['Cloud_pct'] = 1 - frame['Cloud_cloudy_pct']
    frame['Cloud_pct_sq'] = frame['Cloud_pct'] ** 2
    return frame


regression_panel = add_apparent_temperature(regression_panel)
regression_panel = add_weather_percentiles_consistent(regression_panel)

WEATHER_FEATURE_COLUMNS = [
    'apparent_temp',
    'AT_percentile',
    'AT_percentile_sq',
    'Precip_pct',
    'Precip_pct_sq',
    'Cloud_pct',
    'Cloud_pct_sq',
]

feature_missingness = regression_panel[WEATHER_FEATURE_COLUMNS].isna().mean().reset_index()
feature_missingness.columns = ['feature', 'missing_rate']
display(feature_missingness)

## Descriptive tables and figures

The first output reproduces the original descriptive-statistics table using `descriptive_sales_weather_panel.parquet`, which preserves `NettoKr`, `AntallArtikler`, and `AntallSalg` for reporting. The later diagnostic table summarizes the realised-weather regression panel and is not a replacement for the original descriptive table.


In [ ]:
require_file(DESCRIPTIVE_PANEL_PATH, 'descriptive sales/weather reporting panel from notebook 01')

original_descriptive_columns = [
    'DatoSolgt',
    'AvdelingID',
    'Analyse_Kategori',
    'NettoKr',
    'AntallArtikler',
    'AntallSalg',
    'temp_mean',
    'precip_val',
]
descriptive_panel = pd.read_parquet(DESCRIPTIVE_PANEL_PATH, columns=original_descriptive_columns)
require_columns(descriptive_panel, original_descriptive_columns, 'descriptive_panel')
descriptive_panel['DatoSolgt'] = pd.to_datetime(descriptive_panel['DatoSolgt'])

report_memory(descriptive_panel, 'descriptive_panel for original Hovedfilen cell 17')


def get_hovedfilen_descriptive_stats(frame, category_name):
    subset = frame.loc[frame['Analyse_Kategori'] == category_name]
    if len(subset) == 0:
        return pd.DataFrame()
    stats_table = subset[['NettoKr', 'AntallArtikler', 'AntallSalg']].describe().transpose()
    stats_table['Kategori'] = category_name
    return stats_table


hovedfilen_categories = descriptive_panel['Analyse_Kategori'].dropna().unique().tolist()
hovedfilen_stats_parts = []
for category in hovedfilen_categories:
    category_stats = get_hovedfilen_descriptive_stats(descriptive_panel, category)
    if not category_stats.empty:
        hovedfilen_stats_parts.append(category_stats)

if not hovedfilen_stats_parts:
    raise ValueError('No descriptive statistics could be computed from descriptive_panel.')

hovedfilen_summary_stats = pd.concat(hovedfilen_stats_parts)
hovedfilen_summary_stats = hovedfilen_summary_stats.reset_index().rename(columns={'index': 'Variabel'})
hovedfilen_summary_stats = hovedfilen_summary_stats[
    ['Kategori', 'Variabel', 'count', 'mean', 'std', 'min', '50%', 'max']
]

hovedfilen_summary_stats_display = format_dataframe_for_display(
    hovedfilen_summary_stats,
    {
        'count': 'count',
        'mean': 2,
        'std': 2,
        'min': 2,
        '50%': 2,
        'max': 2,
    },
)

save_table_outputs(
    hovedfilen_summary_stats_display,
    ORIGINAL_DESCRIPTIVE_CSV_PATH,
    ORIGINAL_DESCRIPTIVE_HTML_PATH,
    title='DESKRIPTIV STATISTIKK FOR NYE PRODUKTKATEGORIER',
    note=(
        'Notes: Reproduces the original Hovedfilen.ipynb cell 17 descriptive-statistics display. '
        'Statistics are calculated by Analyse_Kategori for NettoKr, AntallArtikler, and AntallSalg. '
        'The input is the separate descriptive reporting panel from notebook 01, not the core modelling panel.'
    ),
    orientation='landscape',
    table_class='tbl-main',
    index=False,
)

missing_rows = []
for category in hovedfilen_categories:
    subset = descriptive_panel.loc[descriptive_panel['Analyse_Kategori'] == category]
    missing_count = int(subset[['NettoKr', 'AntallArtikler', 'temp_mean', 'precip_val']].isna().sum().sum())
    missing_rows.append(
        {
            'Kategori': category,
            'missing_values': missing_count,
            'rows': len(subset),
        }
    )

hovedfilen_missing_values = pd.DataFrame(missing_rows)
hovedfilen_missing_values_display = format_dataframe_for_display(
    hovedfilen_missing_values,
    {'missing_values': 'count', 'rows': 'count'},
)
save_table_outputs(
    hovedfilen_missing_values_display,
    ORIGINAL_DESCRIPTIVE_MISSING_CSV_PATH,
    ORIGINAL_DESCRIPTIVE_MISSING_HTML_PATH,
    title='HOVEDFILEN CELL 17 - Missing Values by Category',
    note=(
        'Notes: Reproduces the missing-value check printed by Hovedfilen.ipynb cell 17 for '
        'NettoKr, AntallArtikler, temp_mean, and precip_val.'
    ),
    orientation='portrait',
    table_class='tbl-main',
    index=False,
)
del descriptive_panel
gc.collect()

# Diagnostic panel summary; separate from the reproduced original descriptive table.
diagnostic_descriptive_source = regression_panel.loc[open_category_mask, [
    'Analyse_Kategori',
    'Antall_total',
    'log_Salg',
    'CampaignDay',
    'temp_mean',
    'apparent_temp',
    'precip_val',
    'wind_mean',
    'cloud_mean',
]]

descriptive_by_category = (
    diagnostic_descriptive_source.groupby('Analyse_Kategori', observed=True)
    .agg(
        rows=('Antall_total', 'size'),
        mean_qty=('Antall_total', 'mean'),
        median_qty=('Antall_total', 'median'),
        zero_qty_rate=('Antall_total', lambda values: float(values.eq(0).mean())),
        mean_log_sales=('log_Salg', 'mean'),
        campaign_day_rate=('CampaignDay', 'mean'),
        mean_temp=('temp_mean', 'mean'),
        mean_apparent_temp=('apparent_temp', 'mean'),
        mean_precip=('precip_val', 'mean'),
        mean_wind=('wind_mean', 'mean'),
        mean_cloud=('cloud_mean', 'mean'),
    )
    .reset_index()
)

descriptive_by_category_output = descriptive_by_category.rename(
    columns={
        'Analyse_Kategori': 'Category',
        'rows': 'Rows',
        'mean_qty': 'Mean quantity',
        'median_qty': 'Median quantity',
        'zero_qty_rate': 'Zero-sales rate',
        'mean_log_sales': 'Mean log sales',
        'campaign_day_rate': 'Campaign-day rate',
        'mean_temp': 'Mean temp',
        'mean_apparent_temp': 'Mean apparent temp',
        'mean_precip': 'Mean precip',
        'mean_wind': 'Mean wind',
        'mean_cloud': 'Mean cloud',
    }
)
descriptive_by_category_output = format_dataframe_for_display(
    descriptive_by_category_output,
    {
        'Rows': 'count',
        'Mean quantity': 2,
        'Median quantity': 2,
        'Zero-sales rate': 3,
        'Mean log sales': 3,
        'Campaign-day rate': 3,
        'Mean temp': 2,
        'Mean apparent temp': 2,
        'Mean precip': 2,
        'Mean wind': 2,
        'Mean cloud': 2,
    },
)
save_table_outputs(
    descriptive_by_category_output,
    DESCRIPTIVE_CSV_PATH,
    DESCRIPTIVE_HTML_PATH,
    title=(
        'DIAGNOSTIC APPENDIX TABLE R.1 - Descriptive statistics by category '
        'for the realised-weather regression panel'
    ),
    note=(
        'Notes: Diagnostic table only. The sample excludes closed store-days and the Ekskluderes category. '
        'Zero-sales rate is the share of open store-category-day observations with Antall_total equal to zero. '
        'Weather variables are realised daily observations.'
    ),
    orientation='landscape',
    table_class='tbl-main',
    index=False,
)

weather_store_day = regression_panel.drop_duplicates(subset=['DatoSolgt', 'AvdelingID'])[WEATHER_COLUMNS]
weather_corr = weather_store_day.corr()
weather_corr.to_csv(WEATHER_CORR_CSV_PATH)



display(hovedfilen_summary_stats_display)
display(hovedfilen_missing_values_display)
display(descriptive_by_category_output)
print(f'Saved: {project_relative(ORIGINAL_DESCRIPTIVE_HTML_PATH)}')
print(f'Saved: {project_relative(DESCRIPTIVE_CSV_PATH)}')
if RENDER_REGRESSION_FIGURES:
    print(f'Figure rendering enabled; expected figure path: {project_relative(WEATHER_CORR_FIG_PATH)}')
    print(f'Figure rendering enabled; expected figure path: {project_relative(LOG_DIST_FIG_PATH)}')
else:
    print('Regression figure rendering skipped (RENDER_REGRESSION_FIGURES=False).')


## Category-level regression sample

The canonical category-level regression sample uses open store-days and retains genuine zero category-sales rows: `Closed == 0`, `Avdeling_total > 0`, `Analyse_Kategori` in `ANALYSIS_CATEGORIES`, and no `Antall_total > 0` filter. The older positive-sales-only sample is not the final thesis methodology.


In [ ]:
SEASON_MAP = {
    'Vår': 'Spring',
    'Sommer': 'Summer',
    'Høst': 'Fall',
    'Vinter': 'Winter',
    'Vår': 'Spring',
    'Höst': 'Fall',
}

CATEGORY_REQUIRED_MODEL_COLUMNS = [
    'log_Salg',
    'Antall_total',
    'Avdeling_total',
    'Weekday',
    'Season',
    'StoreID',
    'Helligdager',
    'CampaignDay',
    'AT_percentile',
    'AT_percentile_sq',
    'Precip_pct',
    'Precip_pct_sq',
    'Cloud_pct',
    'Cloud_pct_sq',
]


def prepare_category_regression_panel(panel):
    d = panel.loc[
        panel['Closed'].eq(0)
        & panel['Avdeling_total'].gt(0)
        & panel['Analyse_Kategori'].isin(ANALYSIS_CATEGORIES)
    ].copy()
    d['StoreID'] = d['AvdelingID'].astype(str)
    d['Weekday'] = d['Ukedag'].astype(str)
    d['Season'] = d['Årstid'].astype('string').replace(SEASON_MAP).astype(str)
    d['Analyse_Kategori'] = d['Analyse_Kategori'].astype(str)
    d['Helligdager'] = pd.to_numeric(d['Helligdager'], errors='coerce').fillna(0).astype(int)
    numeric_columns = [
        'log_Salg',
        'Antall_total',
        'Avdeling_total',
        'CampaignDay',
        'AT_percentile',
        'AT_percentile_sq',
        'Precip_pct',
        'Precip_pct_sq',
        'Cloud_pct',
        'Cloud_pct_sq',
    ]
    for column in numeric_columns:
        d[column] = pd.to_numeric(d[column], errors='coerce')
    d = d.replace([np.inf, -np.inf], np.nan).dropna(subset=CATEGORY_REQUIRED_MODEL_COLUMNS)
    return d


category_regression_panel = prepare_category_regression_panel(regression_panel)
if WRITE_LEGACY_POSITIVE_SALES_OUTPUTS:
    category_regression_positive_sales_legacy_panel = category_regression_panel.loc[
        category_regression_panel['Antall_total'] > 0
    ].copy()
else:
    category_regression_positive_sales_legacy_panel = pd.DataFrame()

report_memory(category_regression_panel, 'category_regression_panel_zero_retaining')
if WRITE_LEGACY_POSITIVE_SALES_OUTPUTS:
    report_memory(category_regression_positive_sales_legacy_panel, 'category_regression_legacy_positive_sales_only')

category_sample_summary = (
    category_regression_panel.groupby('Analyse_Kategori', observed=True)
    .agg(
        rows=('Antall_total', 'size'),
        open_store_days=('Avdeling_total', lambda values: int(values.gt(0).sum())),
        positive_rows=('Antall_total', lambda values: int(values.gt(0).sum())),
        zero_qty_rate=('Antall_total', lambda values: float(values.eq(0).mean())),
        mean_qty=('Antall_total', 'mean'),
        mean_log_sales=('log_Salg', 'mean'),
    )
    .reset_index()
)
display(category_sample_summary)


## Regression formulas and helpers

The model ladder contains the control specification, linear weather terms, nonlinear weather terms, seasonal weather interactions, and the complete seasonal nonlinear specification.


In [ ]:
CAMPAIGN_CTRL = 'CampaignDay'
SEASON_TERM = 'C(Season, Treatment(reference="Spring"))'

base_controls = f'C(Weekday) + C(Season) + C(StoreID) + Helligdager + {CAMPAIGN_CTRL}'
base_controls_complete = f'C(Weekday) + C(StoreID) + Helligdager + {CAMPAIGN_CTRL}'

formula_control = f'log_Salg ~ {base_controls}'
formula_weather = f'log_Salg ~ AT_percentile + Precip_pct + Cloud_pct + {base_controls}'
formula_weather_nl = (
    'log_Salg ~ AT_percentile + AT_percentile_sq + '
    'Precip_pct + Precip_pct_sq + '
    f'Cloud_pct + Cloud_pct_sq + {base_controls}'
)
formula_weather_seasonal = (
    f'log_Salg ~ {SEASON_TERM}*(AT_percentile + Precip_pct + Cloud_pct) + '
    f'{base_controls_complete}'
)
formula_weather_complete = (
    f'log_Salg ~ {SEASON_TERM}*(AT_percentile + AT_percentile_sq + '
    f'Precip_pct + Precip_pct_sq + Cloud_pct + Cloud_pct_sq) + {base_controls_complete}'
)

CATEGORY_MODEL_SPECS = [
    ('(i) Control', formula_control),
    ('(ii) Weather', formula_weather),
    ('(iii) Weather (non-linear)', formula_weather_nl),
    ('(iv) Weather (seasonal)', formula_weather_seasonal),
    ('(v) Weather (complete)', formula_weather_complete),
]

ROWS_TO_SHOW = [
    ('Temperature', 'AT_percentile'),
    ('Precipitation', 'Precip_pct'),
    ('Cloud', 'Cloud_pct'),
    ('Temperature²', 'AT_percentile_sq'),
    ('Precipitation²', 'Precip_pct_sq'),
    ('Cloud²', 'Cloud_pct_sq'),
]


def positive_sales_mape(result, data, outcome_column='Antall_total'):
    positive = data.loc[data[outcome_column] > 0]
    if positive.empty:
        return np.nan
    y_true = positive[outcome_column].astype(float).to_numpy()
    yhat = np.expm1(result.predict(positive))
    eps = 1e-9
    return float(np.mean(np.abs((y_true - yhat) / np.maximum(y_true, eps))) * 100)


class SimpleTestResult:
    def __init__(self, statistic, p_value):
        self.statistic = statistic
        self.fvalue = statistic
        self.pvalue = p_value


def solve_gauss_jordan(matrix, rhs, tol=1e-10):
    a = np.array(matrix, dtype='float64', copy=True)
    b = np.array(rhs, dtype='float64', copy=True)
    one_dimensional = b.ndim == 1
    if one_dimensional:
        b = b.reshape(-1, 1)
    n = a.shape[0]
    if a.shape[0] != a.shape[1] or b.shape[0] != n:
        raise ValueError('solve_gauss_jordan requires a square matrix and compatible rhs.')
    for i in range(n):
        pivot = i + int(np.argmax(np.abs(a[i:, i])))
        if abs(a[pivot, i]) < tol:
            raise ValueError(f'Near-singular OLS normal-equation matrix at pivot {i}.')
        if pivot != i:
            a[[i, pivot], :] = a[[pivot, i], :]
            b[[i, pivot], :] = b[[pivot, i], :]
        pivot_value = a[i, i]
        a[i, :] = a[i, :] / pivot_value
        b[i, :] = b[i, :] / pivot_value
        for j in range(n):
            if j == i:
                continue
            factor = a[j, i]
            if factor != 0:
                a[j, :] -= factor * a[i, :]
                b[j, :] -= factor * b[i, :]
    return b.ravel() if one_dimensional else b


def matrix_product(left, right):
    return np.einsum('ij,jk->ik', left, right, optimize=False)


class SafeOLSResult:
    def __init__(self, formula, y_name, design_info, columns, beta, cov, y, fitted, residuals, df_model, cov_type):
        self.formula = formula
        self.y_name = y_name
        self.design_info = design_info
        self.columns = list(columns)
        self.params = pd.Series(beta, index=self.columns, dtype='float64')
        self._cov = pd.DataFrame(cov, index=self.columns, columns=self.columns)
        self.nobs = int(len(y))
        self.df_model = float(df_model)
        self.df_resid = float(self.nobs - len(self.columns))
        self.resid = pd.Series(residuals, dtype='float64')
        self.fittedvalues = pd.Series(fitted, dtype='float64')
        rss = float(np.sum(residuals**2))
        centered = y - float(np.mean(y))
        tss = float(np.sum(centered**2))
        self.ssr = rss
        self.centered_tss = tss
        self.rsquared = 1 - rss / tss if tss > 0 else np.nan
        self.rsquared_adj = (
            1 - (rss / self.df_resid) / (tss / (self.nobs - 1))
            if tss > 0 and self.df_resid > 0
            else np.nan
        )
        sigma2_mle = max(rss / self.nobs, np.finfo(float).tiny)
        self.llf = -0.5 * self.nobs * (np.log(2 * np.pi) + 1 + np.log(sigma2_mle))
        self.aic = -2 * self.llf + 2 * len(self.columns)
        self.bic = -2 * self.llf + np.log(self.nobs) * len(self.columns)
        se = np.sqrt(np.maximum(np.diag(cov), 0))
        self.bse = pd.Series(se, index=self.columns, dtype='float64')
        t_values = np.divide(beta, se, out=np.full_like(beta, np.nan), where=se > 0)
        self.tvalues = pd.Series(t_values, index=self.columns, dtype='float64')
        if cov_type == 'cluster':
            p_values = 2 * stats.norm.sf(np.abs(t_values))
        else:
            p_values = 2 * stats.t.sf(np.abs(t_values), df=self.df_resid)
        self.pvalues = pd.Series(p_values, index=self.columns, dtype='float64')

    def cov_params(self):
        return self._cov

    def predict(self, data):
        from patsy import build_design_matrices

        design = build_design_matrices([self.design_info], data, return_type='dataframe')[0]
        x = np.asarray(design[self.columns].values, dtype='float64', order='C')
        beta = self.params.to_numpy(dtype='float64')
        return pd.Series(np.einsum('ni,i->n', x, beta, optimize=False), index=data.index)

    def _wald_statistic(self, restriction):
        r = np.asarray(restriction, dtype='float64')
        if r.ndim == 1:
            r = r.reshape(1, -1)
        rb = np.einsum('ij,j->i', r, self.params.to_numpy(dtype='float64'), optimize=False)
        middle = matrix_product(matrix_product(r, self._cov.to_numpy(dtype='float64')), r.T)
        solved = solve_gauss_jordan(middle, rb)
        return float(np.einsum('i,i->', rb, solved, optimize=False)), r.shape[0]

    def wald_test(self, restriction, scalar=True):
        statistic, q = self._wald_statistic(restriction)
        p_value = float(stats.chi2.sf(statistic, q))
        return SimpleTestResult(statistic, p_value)

    def f_test(self, restriction):
        statistic, q = self._wald_statistic(restriction)
        f_value = statistic / q
        p_value = float(stats.f.sf(f_value, q, self.df_resid))
        return SimpleTestResult(f_value, p_value)

    def compare_f_test(self, restricted):
        df_diff = restricted.df_resid - self.df_resid
        if df_diff <= 0:
            return np.nan, np.nan, df_diff
        numerator = (restricted.ssr - self.ssr) / df_diff
        denominator = self.ssr / self.df_resid
        f_value = numerator / denominator if denominator > 0 else np.nan
        p_value = float(stats.f.sf(f_value, df_diff, self.df_resid)) if np.isfinite(f_value) else np.nan
        return f_value, p_value, df_diff


def fit_ols_formula(formula, data, cluster_column=None):
    y_frame, design = dmatrices(formula, data=data, return_type='dataframe')
    y = np.asarray(y_frame.iloc[:, 0].values, dtype='float64')
    x = np.asarray(design.values, dtype='float64', order='C')
    columns = design.columns.tolist()
    xtx = np.einsum('ni,nj->ij', x, x, optimize=False)
    xty = np.einsum('ni,n->i', x, y, optimize=False)
    xtx_inv = solve_gauss_jordan(xtx, np.eye(xtx.shape[0]))
    beta = np.einsum('ij,j->i', xtx_inv, xty, optimize=False)
    fitted = np.einsum('ni,i->n', x, beta, optimize=False)
    residuals = y - fitted
    n, k = x.shape
    if cluster_column is None:
        sigma2 = float(np.sum(residuals**2) / max(n - k, 1))
        cov = sigma2 * xtx_inv
        cov_type = 'nonrobust'
    else:
        groups = data.loc[design.index, cluster_column]
        codes, _uniques = pd.factorize(groups, sort=False)
        n_groups = int(codes.max() + 1)
        cluster_scores = np.zeros((n_groups, k), dtype='float64')
        weighted_x = x * residuals[:, None]
        np.add.at(cluster_scores, codes, weighted_x)
        meat = np.einsum('gi,gj->ij', cluster_scores, cluster_scores, optimize=False)
        cov = matrix_product(matrix_product(xtx_inv, meat), xtx_inv)
        if n_groups > 1 and n > k:
            correction = (n_groups / (n_groups - 1)) * ((n - 1) / (n - k))
            cov = correction * cov
        cov_type = 'cluster'
    return SafeOLSResult(
        formula=formula,
        y_name=y_frame.columns[0],
        design_info=design.design_info,
        columns=columns,
        beta=beta,
        cov=cov,
        y=y,
        fitted=fitted,
        residuals=residuals,
        df_model=k - 1,
        cov_type=cov_type,
    )


def fit_model_ladder(data, model_specs, cluster_column='StoreID'):
    fits_plain = {}
    fits_cluster = {}
    for model_name, formula in model_specs:
        fitted = fit_ols_formula(formula, data, cluster_column=cluster_column)
        fits_plain[model_name] = fitted
        fits_cluster[model_name] = fitted
    return fits_plain, fits_cluster


def build_original_category_ladder_table(fits_plain, fits_cluster, data, rows_to_show, outcome_column='Antall_total'):
    columns = list(fits_plain.keys())
    coef_panel = pd.DataFrame(index=[row[0] for row in rows_to_show], columns=columns)
    for label, term in rows_to_show:
        for column in columns:
            coef_panel.loc[label, column] = format_regression_cell(fits_cluster[column], term)

    fit_panel = pd.DataFrame(
        index=['Controls', 'Observations', 'AIC/BIC', 'MAPE (%)', 'Adjusted R²', 'Δ Adjusted R² (pp)'],
        columns=columns,
    )
    adj_r2_control = fits_plain['(i) Control'].rsquared_adj
    for column in columns:
        model = fits_plain[column]
        fit_panel.loc['Controls', column] = 'Yes'
        fit_panel.loc['Observations', column] = f'{int(model.nobs):,}'
        fit_panel.loc['AIC/BIC', column] = f'{model.aic:,.0f} / {model.bic:,.0f}'
        fit_panel.loc['MAPE (%)', column] = f'{positive_sales_mape(model, data, outcome_column):.1f}'
        fit_panel.loc['Adjusted R²', column] = f'{100 * model.rsquared_adj:.1f}%'
        fit_panel.loc['Δ Adjusted R² (pp)', column] = f'{100 * (model.rsquared_adj - adj_r2_control):.1f}'
    return pd.concat([coef_panel, fit_panel]).where(lambda frame: pd.notna(frame), '')


## Weather-window diagnostics and specification choice

The regression analysis retains `trade_08_19` as the main trading-hours realised-weather window and `full_day_00_23` as the full-day sensitivity benchmark. The redundant `trade_08_18` window is excluded from loops, per-window tables, and diagnostics.


In [ ]:
# Optional diagnostics are disabled by default to avoid kernel crashes.
# Enable only when rebuilding weather-window appendix diagnostics.
RUN_SALES_WINDOW_DIAGNOSTIC = False
RUN_WINDOW_REGRESSION_COMPARISON = False
WET_THRESHOLD_MM = 0.1
WINDOW_HOUR_RANGES = {
    'full_day_00_23': (0, 23),
    'trade_08_19': (8, 19),
}
TABLE_W_INTERNAL_COLUMNS = [
    'window',
    'demand_unit_share',
    'order_share',
    'h2_mean_coverage',
    'h2_full_coverage',
    'h2_precip_rmse',
    'h2_precip_mae',
    'h2_precip_miss_rate',
    'delta_adj_r2_pp',
    'rmse_log',
    'mae_log',
    'recommended_role',
]
TABLE_W_DISPLAY_LABELS = {
    'window': 'Window',
    'demand_unit_share': 'Demand-unit share',
    'order_share': 'Order share',
    'h2_mean_coverage': 'h=2 mean coverage',
    'h2_full_coverage': 'h=2 full coverage',
    'h2_precip_rmse': 'h=2 precip. RMSE',
    'h2_precip_mae': 'h=2 precip. MAE',
    'h2_precip_miss_rate': 'h=2 precip. miss rate',
    'delta_adj_r2_pp': 'Delta Adj. R2 (pp)',
    'rmse_log': 'RMSE log',
    'mae_log': 'MAE log',
    'recommended_role': 'Recommended role',
}


def extract_hour_from_timeid(series):
    values = series.dropna()
    if values.empty:
        return pd.Series(np.nan, index=series.index, dtype='float64')
    if pd.api.types.is_numeric_dtype(values):
        numeric = pd.to_numeric(series, errors='coerce')
        max_value = numeric.max(skipna=True)
        if pd.notna(max_value) and max_value <= 23:
            return numeric.astype('float64')
        return np.floor(numeric / 100).astype('float64')
    parsed = pd.to_datetime(series.astype(str), errors='coerce')
    return parsed.dt.hour.astype('float64')


def choose_first_available(available_columns, candidates):
    for column in candidates:
        if column in available_columns:
            return column
    return None


def load_sales_by_hour(dataset_path):
    if not dataset_path.exists():
        print(f'Dataset.parquet not found at {project_relative(dataset_path)}; sales-hour diagnostic skipped.')
        return pd.DataFrame(), 'Unavailable'
    try:
        import pyarrow.dataset as ds

        dataset = ds.dataset(dataset_path, format='parquet')
        available_columns = set(dataset.schema.names)
        time_col = choose_first_available(
            available_columns,
            ['TimeID', 'timeid', 'Time', 'hour', 'Hour', 'Klokkeslett'],
        )
        if time_col is None:
            print('Could not find TimeID/hour column; sales-hour diagnostic skipped.')
            return pd.DataFrame(), 'Unavailable'

        demand_unit_col = choose_first_available(available_columns, ['Antall_total', 'AntallArtikler'])
        columns = [time_col]
        for column in [demand_unit_col, 'OrdreID', 'NettoKr']:
            if column and column in available_columns and column not in columns:
                columns.append(column)

        chunks = []
        for batch in dataset.scanner(columns=columns, batch_size=500_000).to_batches():
            part = batch.to_pandas()
            part['hour'] = extract_hour_from_timeid(part[time_col])
            part = part.dropna(subset=['hour'])
            if part.empty:
                continue
            aggregations = {}
            if demand_unit_col is not None:
                part[demand_unit_col] = pd.to_numeric(part[demand_unit_col], errors='coerce').fillna(0)
                aggregations['demand_units'] = (demand_unit_col, 'sum')
            if 'NettoKr' in part.columns:
                part['NettoKr'] = pd.to_numeric(part['NettoKr'], errors='coerce').fillna(0)
                aggregations['net_sales'] = ('NettoKr', 'sum')
            if aggregations:
                grouped = part.groupby('hour', observed=True).agg(**aggregations).reset_index()
            else:
                grouped = part.groupby('hour', observed=True).size().reset_index(name='rows')
            if 'OrdreID' in part.columns:
                grouped_orders = part.groupby('hour', observed=True)['OrdreID'].nunique().reset_index(name='orders')
                grouped = grouped.merge(grouped_orders, on='hour', how='left')
            chunks.append(grouped)
        if not chunks:
            return pd.DataFrame(), 'Unavailable'
        sales_hour = (
            pd.concat(chunks, ignore_index=True)
            .groupby('hour', observed=True)
            .sum(numeric_only=True)
            .reset_index()
        )
        sales_hour['hour'] = sales_hour['hour'].astype(int)
        if 'demand_units' not in sales_hour.columns:
            if 'orders' in sales_hour.columns:
                sales_hour['demand_units'] = sales_hour['orders']
                demand_unit_col = 'order count fallback'
            elif 'net_sales' in sales_hour.columns:
                sales_hour['demand_units'] = sales_hour['net_sales']
                demand_unit_col = 'NettoKr fallback'
            else:
                return pd.DataFrame(), 'Unavailable'
        return sales_hour.sort_values('hour').reset_index(drop=True), demand_unit_col or 'Unavailable'
    except Exception as exc:
        print(f'Sales-hour diagnostic failed: {exc}')
        return pd.DataFrame(), 'Unavailable'


def shortest_interval_for_share(hour_frame, value_column='demand_units', target_share=0.95):
    if hour_frame.empty or value_column not in hour_frame.columns or hour_frame[value_column].sum() <= 0:
        return ''
    best = None
    hours = sorted(hour_frame['hour'].unique())
    total = hour_frame[value_column].sum()
    by_hour = hour_frame.set_index('hour')[value_column]
    for start in hours:
        for end in hours:
            if end < start:
                continue
            share = by_hour.loc[(by_hour.index >= start) & (by_hour.index <= end)].sum() / total
            if share >= target_share:
                candidate = (end - start, start, end, share)
                if best is None or candidate < best:
                    best = candidate
    return f'{int(best[1]):02d}-{int(best[2]):02d}' if best else ''


def require_internal_columns(frame, required_columns, frame_name):
    missing = [column for column in required_columns if column not in frame.columns]
    if missing:
        print(f'{frame_name} columns: {frame.columns.tolist()}')
        raise KeyError(f'{frame_name} is missing required internal columns: {missing}')


def regression_window_role(window):
    return {
        'full_day_00_23': 'Full-day sensitivity benchmark',
        'trade_08_19': 'Main trading-hours window',
    }[window]


# Quantity is the timing measure because the dependent variable is Antall_total.
sales_hour, demand_unit_measure = (
    load_sales_by_hour(DATASET_PATH)
    if RUN_SALES_WINDOW_DIAGNOSTIC
    else (pd.DataFrame(), 'Unavailable')
)
demand_rows = []
if sales_hour.empty:
    hours_for_95_demand_units = ''
    for window in WEATHER_WINDOW_ORDER:
        demand_rows.append({'window': window, 'demand_unit_share': np.nan, 'order_share': np.nan})
else:
    total_units = sales_hour['demand_units'].sum()
    total_orders = sales_hour['orders'].sum() if 'orders' in sales_hour.columns else np.nan
    for window in WEATHER_WINDOW_ORDER:
        start, end = WINDOW_HOUR_RANGES[window]
        sub = sales_hour.loc[sales_hour['hour'].between(start, end)]
        demand_rows.append(
            {
                'window': window,
                'demand_unit_share': 100 * sub['demand_units'].sum() / total_units if total_units else np.nan,
                'order_share': (
                    100 * sub['orders'].sum() / total_orders
                    if 'orders' in sales_hour.columns and total_orders
                    else np.nan
                ),
            }
        )
    hours_for_95_demand_units = shortest_interval_for_share(sales_hour, 'demand_units', 0.95)
demand_share_raw = pd.DataFrame(demand_rows)
require_internal_columns(demand_share_raw, ['window', 'demand_unit_share', 'order_share'], 'demand_share_raw')

# h=2 deterministic forecast coverage.
require_file(FORECAST_WINDOWS_PATH, 'deterministic forecast weather window output from notebook 01b')
forecast_windows_diag = pd.read_parquet(
    FORECAST_WINDOWS_PATH,
    columns=['horizon_days', 'aggregation_window', 'coverage_share', 'is_full_window'],
)
h2_windows = forecast_windows_diag.loc[forecast_windows_diag['horizon_days'].eq(2)].copy()
coverage_rows = []
for window in WEATHER_WINDOW_ORDER:
    sub = h2_windows.loc[h2_windows['aggregation_window'].eq(window)]
    coverage_rows.append(
        {
            'window': window,
            'h2_mean_coverage': 100 * sub['coverage_share'].mean(),
            'h2_full_coverage': 100 * sub['is_full_window'].mean(),
        }
    )
coverage_raw = pd.DataFrame(coverage_rows)
require_internal_columns(coverage_raw, ['window', 'h2_mean_coverage', 'h2_full_coverage'], 'coverage_raw')

# h=2 last-available-hour audit context.
require_file(FORECAST_HOUR_AUDIT_PATH, 'forecast horizon-hour audit from notebook 01b')
hour_audit = pd.read_parquet(FORECAST_HOUR_AUDIT_PATH)
h2_available = hour_audit.loc[hour_audit['horizon_days'].eq(2) & hour_audit['available'].astype(bool)].copy()
h2_last_hour = h2_available.groupby(
    ['origin_date', 'origin_hour', 'target_date'],
    observed=True,
)['target_hour_local'].max()
most_common_h2_last_hour = int(h2_last_hour.mode().iloc[0]) if not h2_last_hour.empty else np.nan
origin_count = h2_last_hour.shape[0]
covered_by_hour = (
    h2_available.drop_duplicates(['origin_date', 'origin_hour', 'target_date', 'target_hour_local'])
    .groupby('target_hour_local', observed=True)
    .size()
    / origin_count
    if origin_count
    else pd.Series(dtype='float64')
)
latest_hour_95 = (
    int(covered_by_hour[covered_by_hour.ge(0.95)].index.max())
    if not covered_by_hour.empty and covered_by_hour.ge(0.95).any()
    else np.nan
)
latest_hour_90 = (
    int(covered_by_hour[covered_by_hour.ge(0.90)].index.max())
    if not covered_by_hour.empty and covered_by_hour.ge(0.90).any()
    else np.nan
)
print(f'Most common h=2 last local target hour: {most_common_h2_last_hour}')
print(f'Latest local hour available for >=95% of h=2 rows: {latest_hour_95}')
print(f'Latest local hour available for >=90% of h=2 rows: {latest_hour_90}')

# h=2 precipitation forecast-versus-realised diagnostics.
require_file(REALISED_WINDOWS_PATH, 'realised weather window output from notebook 01a')
forecast_precip = pd.read_parquet(
    FORECAST_WINDOWS_PATH,
    columns=['target_date', 'horizon_days', 'AvdelingID', 'aggregation_window', 'precip_fcst_sum'],
)
realised_precip = pd.read_parquet(
    REALISED_WINDOWS_PATH,
    columns=['date', 'AvdelingID', 'aggregation_window', 'precip_obs_sum'],
).rename(columns={'date': 'target_date'})
forecast_precip['target_date'] = pd.to_datetime(forecast_precip['target_date']).dt.normalize()
realised_precip['target_date'] = pd.to_datetime(realised_precip['target_date']).dt.normalize()
precip_compare = forecast_precip.loc[forecast_precip['horizon_days'].eq(2)].merge(
    realised_precip,
    on=['target_date', 'AvdelingID', 'aggregation_window'],
    how='inner',
)
precip_rows = []
for window in WEATHER_WINDOW_ORDER:
    sub = precip_compare.loc[
        precip_compare['aggregation_window'].eq(window),
        ['precip_fcst_sum', 'precip_obs_sum'],
    ].dropna()
    obs_wet = sub['precip_obs_sum'].ge(WET_THRESHOLD_MM)
    fcst_wet = sub['precip_fcst_sum'].ge(WET_THRESHOLD_MM)
    precip_rows.append(
        {
            'window': window,
            'h2_precip_rmse': (
                float(np.sqrt(np.mean((sub['precip_fcst_sum'] - sub['precip_obs_sum']) ** 2)))
                if len(sub)
                else np.nan
            ),
            'h2_precip_mae': (
                float(np.mean(np.abs(sub['precip_fcst_sum'] - sub['precip_obs_sum'])))
                if len(sub)
                else np.nan
            ),
            'h2_precip_miss_rate': (
                100 * float((obs_wet & ~fcst_wet).sum() / max(obs_wet.sum(), 1))
                if len(sub)
                else np.nan
            ),
        }
    )
precip_raw = pd.DataFrame(precip_rows)
require_internal_columns(
    precip_raw,
    ['window', 'h2_precip_rmse', 'h2_precip_mae', 'h2_precip_miss_rate'],
    'precip_raw',
)

# Explanatory regression comparison across realised-weather windows.
WINDOW_MODEL_COLUMNS = unique_preserving_order(REQUIRED_WINDOW_PANEL_COLUMNS)


def read_candidate_window_panel(path, windows, columns):
    try:
        frame = pd.read_parquet(
            path,
            columns=columns,
            filters=[('aggregation_window', 'in', list(windows))],
        )
    except Exception as exc:
        print(
            f'Filtered parquet read failed for weather-window comparison ({exc}). '
            'Falling back to column-pruned read before filtering.'
        )
        frame_all = pd.read_parquet(path, columns=columns)
        frame = frame_all.loc[frame_all['aggregation_window'].isin(windows)].copy()
        del frame_all
        gc.collect()
    return frame


def prepare_window_model_panel(candidate_window_panel):
    require_columns(
        candidate_window_panel,
        list(WINDOW_WEATHER_COLUMN_MAP.keys()) + ['aggregation_window'],
        'candidate_window_panel',
    )
    d = candidate_window_panel.copy()
    for source, target in WINDOW_WEATHER_COLUMN_MAP.items():
        d[target] = pd.to_numeric(d[source], errors='coerce')
    d['DatoSolgt'] = pd.to_datetime(d['DatoSolgt']).dt.normalize()
    d['AvdelingID'] = pd.to_numeric(d['AvdelingID'], errors='raise').astype('int64')
    d['Antall_total'] = pd.to_numeric(d['Antall_total'], errors='coerce')
    d['log_Salg'] = np.log1p(d['Antall_total'])
    d = add_apparent_temperature(d)
    d = add_weather_percentiles_consistent(d)
    return prepare_category_regression_panel(d)


regression_rows = []
if RUN_WINDOW_REGRESSION_COMPARISON:
    require_file(MASTER_PANEL_WINDOWS_PATH, 'window-aware master panel from notebook 01')
    candidate_window_panel = read_candidate_window_panel(
        MASTER_PANEL_WINDOWS_PATH,
        WEATHER_WINDOW_ORDER,
        WINDOW_MODEL_COLUMNS,
    )
    window_model_panel = prepare_window_model_panel(candidate_window_panel)
    del candidate_window_panel
    gc.collect()
    window_base_controls = f'C(Analyse_Kategori) + {base_controls}'
    window_control_formula = f'log_Salg ~ {window_base_controls}'
    window_weather_formula = (
        'log_Salg ~ AT_percentile + AT_percentile_sq + '
        'Precip_pct + Precip_pct_sq + Cloud_pct + Cloud_pct_sq + '
        f'{window_base_controls}'
    )
    for window in WEATHER_WINDOW_ORDER:
        data_w = window_model_panel.loc[window_model_panel['aggregation_window'].eq(window)]
        if len(data_w) < 100:
            continue
        control_plain = fit_ols_formula(window_control_formula, data_w, cluster_column=None)
        weather_plain = fit_ols_formula(window_weather_formula, data_w, cluster_column=None)
        f_value, f_pvalue, _df_diff = weather_plain.compare_f_test(control_plain)
        residual = data_w['log_Salg'] - weather_plain.predict(data_w)
        regression_rows.append(
            {
                'window': window,
                'observations': int(weather_plain.nobs),
                'adj_r2_control': control_plain.rsquared_adj,
                'adj_r2_weather': weather_plain.rsquared_adj,
                'delta_adj_r2_pp': 100 * (weather_plain.rsquared_adj - control_plain.rsquared_adj),
                'rmse_log': float(np.sqrt(np.mean(residual**2))),
                'mae_log': float(np.mean(np.abs(residual))),
                'weather_f_pvalue': float(f_pvalue),
            }
        )
    del window_model_panel
    gc.collect()
else:
    print('Window regression comparison skipped because RUN_WINDOW_REGRESSION_COMPARISON is False.')

window_regression_raw = pd.DataFrame(regression_rows)
if window_regression_raw.empty:
    window_regression_raw = pd.DataFrame(
        {
            'window': WEATHER_WINDOW_ORDER,
            'delta_adj_r2_pp': np.nan,
            'rmse_log': np.nan,
            'mae_log': np.nan,
        }
    )
require_internal_columns(
    window_regression_raw,
    ['window', 'delta_adj_r2_pp', 'rmse_log', 'mae_log'],
    'window_regression_raw',
)

# Keep internal names ASCII-safe; apply display labels only at export.
window_role_map = {window: regression_window_role(window) for window in WEATHER_WINDOW_ORDER}
window_decision_raw = (
    pd.DataFrame({'window': WEATHER_WINDOW_ORDER})
    .merge(demand_share_raw, on='window', how='left')
    .merge(coverage_raw, on='window', how='left')
    .merge(precip_raw, on='window', how='left')
    .merge(window_regression_raw[['window', 'delta_adj_r2_pp', 'rmse_log', 'mae_log']], on='window', how='left')
)
window_decision_raw['recommended_role'] = window_decision_raw['window'].map(window_role_map)
require_internal_columns(window_decision_raw, TABLE_W_INTERNAL_COLUMNS, 'window_decision_raw')
window_decision_raw = window_decision_raw[TABLE_W_INTERNAL_COLUMNS]

window_decision_display = window_decision_raw.rename(columns=TABLE_W_DISPLAY_LABELS)
window_decision_table = format_dataframe_for_display(
    window_decision_display,
    {
        'Demand-unit share': 'percent',
        'Order share': 'percent',
        'h=2 mean coverage': 'percent',
        'h=2 full coverage': 'percent',
        'h=2 precip. RMSE': 2,
        'h=2 precip. MAE': 2,
        'h=2 precip. miss rate': 'percent',
        'Delta Adj. R2 (pp)': 2,
        'RMSE log': 3,
        'MAE log': 3,
    },
)
decision_note = (
    f'Notes: Demand-unit share is based on {demand_unit_measure}; revenue is not '
    'the primary timing measure because the dependent variable is quantity. '
    'The shortest contiguous interval covering approximately 95% of demand units is '
    f'{hours_for_95_demand_units or "not available"}. '
    f'For h=2 forecasts, the most common last available local target hour is {most_common_h2_last_hour}; '
    f'the latest local hour available for at least 95% and 90% of h=2 rows is {latest_hour_95} and {latest_hour_90}. '
    'Regression metrics compare otherwise identical explanatory specifications by realised-weather window.'
)
save_table_outputs(
    window_decision_table,
    WEATHER_WINDOW_DECISION_CSV_PATH,
    WEATHER_WINDOW_DECISION_HTML_PATH,
    title='Table W: Weather-window diagnostics and specification choice',
    note=decision_note,
    orientation='landscape',
    table_class='tbl-main',
    index=False,
    extra_css='.tbl-main th, .tbl-main td { font-size: 8.6pt; }',
)
display(window_decision_table)


### Thesis interpretation

The weather-window comparison is not causal evidence and should not be reduced to adjusted R2 or log-error metrics alone because the remaining window definitions are related. The specification choice combines demand-unit relevance, h=2 forecast coverage, precipitation forecast diagnostics, and explanatory performance. On this basis, `trade_08_19` is the main trading-hours window, while `full_day_00_23` is retained only as a sensitivity benchmark.


## Detailed category model ladder

The detailed category ladder is produced for `Hot Drinks` by default and exported as CSV/HTML using the updated open-store-day sample with zero category-sales rows retained.


In [ ]:
CATEGORY_FOR_DETAILED_TABLE = 'Hot Drinks'

detailed_category_data = category_regression_panel.loc[
    category_regression_panel['Analyse_Kategori'] == CATEGORY_FOR_DETAILED_TABLE
]
if len(detailed_category_data) < 50:
    raise ValueError(f'Not enough rows for detailed category model: {CATEGORY_FOR_DETAILED_TABLE}')

category_fits_plain, category_fits_cluster = fit_model_ladder(detailed_category_data, CATEGORY_MODEL_SPECS)
category_ladder_table = build_original_category_ladder_table(
    category_fits_plain,
    category_fits_cluster,
    detailed_category_data,
    ROWS_TO_SHOW,
    outcome_column='Antall_total',
)

save_table_outputs(
    category_ladder_table,
    CATEGORY_LADDER_CSV_PATH,
    CATEGORY_LADDER_HTML_PATH,
    title=(
        'TABLE X - Regression results for weather effect on daily sales '
        f'(trading days): {CATEGORY_FOR_DETAILED_TABLE}'
    ),
    note=(
        'Notes: Active updated-methodology category ladder. The dependent variable is log(1 + Antall_total). '
        'The sample uses open store-days (Closed == 0 and Avdeling_total > 0), included analysis categories, '
        'and retains zero category-sales observations. t-statistics in parentheses use '
        'standard errors clustered by store. '
        '*, **, *** denote significance at the 5%, 1%, and 0.1% levels in the original helper.'
    ),
    orientation='landscape',
    table_class='tbl-main',
    index=True,
)
display(category_ladder_table)


## Category complete models and appendix tables

This section creates the canonical category-level thesis tables from the updated zero-retaining sample. `thesis_context/tables/` is used only as a visual/style reference.


In [ ]:
def wald_p(model, term_regex):
    names = model.params.index.tolist()
    idx = [i for i, name in enumerate(names) if re.search(term_regex, name)]
    if not idx:
        return np.nan
    restriction = np.zeros((len(idx), len(names)))
    for row_index, param_index in enumerate(idx):
        restriction[row_index, param_index] = 1.0
    return float(model.wald_test(restriction, scalar=True).pvalue)


def interaction_name(season, variable):
    return f'{SEASON_TERM}[T.{season}]:{variable}'


def season_coef(model, variable, season, baseline='Spring'):
    coef = float(model.params.get(variable, 0.0))
    if season != baseline:
        coef += float(model.params.get(interaction_name(season, variable), 0.0))
    return coef


def block_wald_p_total_season(model, season, linear_var, square_var, baseline='Spring'):
    names = list(model.params.index)
    rows = []

    def add_total_constraint(variable):
        row = np.zeros(len(names))
        if variable in names:
            row[names.index(variable)] = 1.0
        if season != baseline:
            interaction = interaction_name(season, variable)
            if interaction in names:
                row[names.index(interaction)] = 1.0
        rows.append(row)

    add_total_constraint(linear_var)
    add_total_constraint(square_var)
    return float(model.wald_test(np.vstack(rows), scalar=True).pvalue)


def season_total_p(model, season, variable, baseline='Spring'):
    names = list(model.params.index)
    row = np.zeros(len(names))
    if variable in names:
        row[names.index(variable)] = 1.0
    if season != baseline:
        interaction = interaction_name(season, variable)
        if interaction in names:
            row[names.index(interaction)] = 1.0
    return float(model.wald_test(row.reshape(1, -1), scalar=True).pvalue)


def iqr_effect_pct(model, season, linear_var, square_var, p_lo=0.25, p_hi=0.75, x0=0.5, baseline='Spring'):
    b1 = season_coef(model, linear_var, season, baseline)
    b2 = season_coef(model, square_var, season, baseline)
    dlog_hi = b1 * (p_hi - x0) + b2 * (p_hi**2 - x0**2)
    dlog_lo = b1 * (p_lo - x0) + b2 * (p_lo**2 - x0**2)
    return 100 * ((np.exp(dlog_hi) - 1) - (np.exp(dlog_lo) - 1))


def fmt_effect_with_stars(effect, p_value, digits=2):
    if pd.isna(effect):
        return ''
    return f'{effect:.{digits}f}{stars_category_summary(p_value)}'


def significant_weather_vars(p_temp, p_precip, p_cloud, alpha=0.05):
    out = []
    if pd.notna(p_temp) and p_temp < alpha:
        out.append('AT')
    if pd.notna(p_precip) and p_precip < alpha:
        out.append('Precip')
    if pd.notna(p_cloud) and p_cloud < alpha:
        out.append('Cloud')
    return ', '.join(out) if out else 'None'


regex_all_weather = r'(AT_percentile|AT_percentile_sq|Precip_pct|Precip_pct_sq|Cloud_pct|Cloud_pct_sq)'
regex_temp = r'(AT_percentile|AT_percentile_sq)'
regex_precip = r'(Precip_pct|Precip_pct_sq)'
regex_cloud = r'(Cloud_pct|Cloud_pct_sq)'

seasons_display = ['Winter', 'Spring', 'Summer', 'Fall']
baseline_season = 'Spring'
vars_map = {
    'AT': ('AT_percentile', 'AT_percentile_sq'),
    'Precip': ('Precip_pct', 'Precip_pct_sq'),
    'Cloud': ('Cloud_pct', 'Cloud_pct_sq'),
}

complete_models_cluster = {} if RENDER_REGRESSION_FIGURES else None
rows_main = []
rows_meta = []
rows_iqr = []

for category in sorted(category_regression_panel['Analyse_Kategori'].dropna().unique()):
    category_data = category_regression_panel.loc[
        category_regression_panel['Analyse_Kategori'] == category
    ].copy()
    if len(category_data) < 50:
        continue

    control_plain = fit_ols_formula(formula_control, category_data, cluster_column='StoreID')
    complete_cluster = fit_ols_formula(formula_weather_complete, category_data, cluster_column='StoreID')
    complete_plain = complete_cluster
    if RENDER_REGRESSION_FIGURES:
        complete_models_cluster[category] = complete_cluster

    delta_pp = (complete_plain.rsquared_adj - control_plain.rsquared_adj) * 100
    p_all = wald_p(complete_cluster, regex_all_weather)
    p_temp = wald_p(complete_cluster, regex_temp)
    p_precip = wald_p(complete_cluster, regex_precip)
    p_cloud = wald_p(complete_cluster, regex_cloud)

    rows_main.append(
        {
            'Category': category,
            'Ref. season': 'Spring',
            'Adj. R² ctrl': round(control_plain.rsquared_adj, 3),
            'Adj. R² full': round(complete_plain.rsquared_adj, 3),
            'Δ Adj. R²': round(delta_pp, 2),
            'W(all)': stars_category_summary(p_all),
            'Sig. weather vars': significant_weather_vars(p_temp, p_precip, p_cloud, alpha=0.05),
        }
    )

    for season in seasons_display:
        for label, (linear_var, square_var) in vars_map.items():
            p_linear = season_total_p(complete_cluster, season, linear_var, baseline=baseline_season)
            p_square = season_total_p(complete_cluster, season, square_var, baseline=baseline_season)
            rows_meta.append(
                {
                    'Season': season,
                    'Weather var': label,
                    'L_sig': int((p_linear < 0.05)) if np.isfinite(p_linear) else 0,
                    'Q_sig': int((p_square < 0.05)) if np.isfinite(p_square) else 0,
                }
            )

    for season in seasons_display:
        row = {'Category': category, 'Season': season}
        for label, (linear_var, square_var) in vars_map.items():
            p_joint = block_wald_p_total_season(
                complete_cluster,
                season,
                linear_var,
                square_var,
                baseline=baseline_season,
            )
            effect = iqr_effect_pct(
                complete_cluster,
                season,
                linear_var,
                square_var,
                p_lo=0.25,
                p_hi=0.75,
                x0=0.5,
                baseline=baseline_season,
            )
            row[f'{label} IQR effect'] = fmt_effect_with_stars(effect, p_joint, digits=2)
        rows_iqr.append(row)

df_main = pd.DataFrame(rows_main).sort_values('Δ Adj. R²', ascending=False).reset_index(drop=True)
df_meta = pd.DataFrame(rows_meta)

appendix1_season = (
    df_meta.groupby(['Season', 'Weather var'])[['L_sig', 'Q_sig']]
    .mean()
    .reset_index()
    .rename(columns={'L_sig': 'Linear sig. (%)', 'Q_sig': 'Quadratic sig. (%)'})
)
appendix1_season['Linear sig. (%)'] = (appendix1_season['Linear sig. (%)'] * 100).round(1)
appendix1_season['Quadratic sig. (%)'] = (appendix1_season['Quadratic sig. (%)'] * 100).round(1)

appendix1_total = (
    df_meta.groupby('Weather var')[['L_sig', 'Q_sig']]
    .mean()
    .reset_index()
    .rename(columns={'L_sig': 'Linear sig. (%)', 'Q_sig': 'Quadratic sig. (%)'})
)
appendix1_total['Linear sig. (%)'] = (appendix1_total['Linear sig. (%)'] * 100).round(1)
appendix1_total['Quadratic sig. (%)'] = (appendix1_total['Quadratic sig. (%)'] * 100).round(1)

season_order = ['Winter', 'Spring', 'Summer', 'Fall']
SEASONS = season_order  # Shared order for the quadratic weather-effect figure cell.
weather_order = ['AT', 'Precip', 'Cloud']

appendix1_season['Season'] = pd.Categorical(
    appendix1_season['Season'],
    categories=season_order,
    ordered=True,
)
appendix1_season['Weather var'] = pd.Categorical(
    appendix1_season['Weather var'],
    categories=weather_order,
    ordered=True,
)
appendix1_season = appendix1_season.sort_values(['Season', 'Weather var']).reset_index(drop=True)

appendix1_total['Weather var'] = pd.Categorical(
    appendix1_total['Weather var'],
    categories=weather_order,
    ordered=True,
)
appendix1_total = appendix1_total.sort_values('Weather var').reset_index(drop=True)

df_iqr = pd.DataFrame(rows_iqr)
cat_order = df_main['Category'].tolist()
df_iqr['Category'] = pd.Categorical(df_iqr['Category'], categories=cat_order, ordered=True)
df_iqr['Season'] = pd.Categorical(df_iqr['Season'], categories=season_order, ordered=True)
df_iqr = df_iqr.sort_values(['Category', 'Season']).reset_index(drop=True)

df_main.to_csv(CATEGORY_SUMMARY_CSV_PATH, index=False)
appendix1_season.to_csv(SIGNIFICANCE_PANELA_CSV_PATH, index=False)
appendix1_total.to_csv(SIGNIFICANCE_PANELB_CSV_PATH, index=False)
df_iqr.to_csv(IQR_EFFECTS_CSV_PATH, index=False)

css_main = hovedfilen_common_css(page_size='A4 landscape', margin='12mm', body_width='297mm') + '''
<style>
  table.tbl-main th:nth-child(1), table.tbl-main td:nth-child(1) { width: 13%; text-align: left;  white-space: normal; }
  table.tbl-main th:nth-child(2), table.tbl-main td:nth-child(2) { width: 11%; text-align: center; white-space: nowrap; }
  table.tbl-main th:nth-child(3), table.tbl-main td:nth-child(3) { width: 10%; text-align: center; white-space: nowrap; }
  table.tbl-main th:nth-child(4), table.tbl-main td:nth-child(4) { width: 10%; text-align: center; white-space: nowrap; }
  table.tbl-main th:nth-child(5), table.tbl-main td:nth-child(5) { width: 9%;  text-align: center; white-space: nowrap; }
  table.tbl-main th:nth-child(6), table.tbl-main td:nth-child(6) { width: 8%;  text-align: center; white-space: nowrap; }
  table.tbl-main th:nth-child(7), table.tbl-main td:nth-child(7) { width: 19%; text-align: center; white-space: normal; }
  table.tbl-main th:nth-child(8), table.tbl-main td:nth-child(8) { width: 12%; text-align: center; white-space: nowrap; }
</style>
'''

title_main = 'TABLE 3 - Category-level summary of weather effects on daily sales'
note_main = (
    'Notes: The table compares the control model with the complete weather specification at the category level. '
    'The sample uses open store-days (Closed == 0 and Avdeling_total > 0), included analysis categories, '
    'and retains zero category-sales observations where Antall_total == 0. W(all) reports the joint significance '
    'of all weather terms. Sig. weather vars are based on joint Wald tests for AT, Precip, and Cloud. '
    'Sig. season terms counts the total number of significant weather terms across seasons. '
    '*, **, *** denote significance at the 10%, 5%, and 1% levels.'
)
html_main = (
    '<html><head>' + css_main + '</head><body>'
    + '<div class="page">'
    + f'<div class="title">{title_main}</div>'
    + html_table_no_index(df_main, 'tbl-main')
    + f'<div class="note">{note_main}</div>'
    + '</div></body></html>'
)
CATEGORY_SUMMARY_HTML_PATH.write_text(html_main, encoding='utf-8')

css_app1 = hovedfilen_common_css(page_size='A4 portrait', margin='16mm') + '''
<style>
  table.tbl-app1a th:nth-child(1), table.tbl-app1a td:nth-child(1) { width: 22%; text-align: left;  white-space: nowrap; }
  table.tbl-app1a th:nth-child(2), table.tbl-app1a td:nth-child(2) { width: 22%; text-align: center; white-space: nowrap; }
  table.tbl-app1a th:nth-child(3), table.tbl-app1a td:nth-child(3) { width: 28%; text-align: center; white-space: nowrap; }
  table.tbl-app1a th:nth-child(4), table.tbl-app1a td:nth-child(4) { width: 28%; text-align: center; white-space: nowrap; }

  table.tbl-app1b th:nth-child(1), table.tbl-app1b td:nth-child(1) { width: 30%; text-align: left;  white-space: nowrap; }
  table.tbl-app1b th:nth-child(2), table.tbl-app1b td:nth-child(2) { width: 35%; text-align: center; white-space: nowrap; }
  table.tbl-app1b th:nth-child(3), table.tbl-app1b td:nth-child(3) { width: 35%; text-align: center; white-space: nowrap; }
</style>
'''
title_app1 = 'TABLE A.2 - Share of significant weather terms across seasons and overall'
note_app1 = (
    'Notes: Panel A reports the share of category-season combinations in which the '
    'linear or quadratic term for a given '
    'weather variable is significant at the 5 percent level, by season. Panel B reports the same shares overall. '
    'The sample uses open store-days and retains zero category-sales observations.'
)
html_app1 = (
    '<html><head>' + css_app1 + '</head><body>'
    + '<div class="page">'
    + f'<div class="title">{title_app1}</div>'
    + '<div class="subtitle">Panel A. Share by season</div>'
    + html_table_no_index(appendix1_season, 'tbl-app1a')
    + '<div class="rule"></div>'
    + '<div class="subtitle">Panel B. Overall share across category-season combinations</div>'
    + html_table_no_index(appendix1_total, 'tbl-app1b')
    + f'<div class="note">{note_app1}</div>'
    + '</div></body></html>'
)
SIGNIFICANCE_HTML_PATH.write_text(html_app1, encoding='utf-8')

css_app2 = hovedfilen_common_css(page_size='A4 landscape', margin='12mm', body_width='297mm') + '''
<style>
  table.tbl-app2 th:nth-child(1), table.tbl-app2 td:nth-child(1) { width: 18%; text-align: left;  white-space: nowrap; }
  table.tbl-app2 th:nth-child(2), table.tbl-app2 td:nth-child(2) { width: 12%; text-align: center; white-space: nowrap; }
  table.tbl-app2 th:nth-child(3), table.tbl-app2 td:nth-child(3) { width: 23%; text-align: center; white-space: nowrap; }
  table.tbl-app2 th:nth-child(4), table.tbl-app2 td:nth-child(4) { width: 23%; text-align: center; white-space: nowrap; }
  table.tbl-app2 th:nth-child(5), table.tbl-app2 td:nth-child(5) { width: 24%; text-align: center; white-space: nowrap; }
</style>
'''
title_app2 = 'TABLE A.3 - Seasonal interquartile-range effects of weather variables by category'
note_app2 = (
    'Notes: Reported values show the estimated change in sales associated with a '
    'shift from the 25th to the 75th percentile '
    'of each weather variable within a given season, based on the complete weather '
    'specification. The sample uses open store-days '
    'and retains zero category-sales observations. Significance stars are based on '
    'a joint season-specific Wald test for the linear '
    'and quadratic term of each weather variable. *, **, *** denote significance at the 10%, 5%, and 1% levels.'
)
html_app2 = (
    '<html><head>' + css_app2 + '</head><body>'
    + '<div class="page">'
    + f'<div class="title">{title_app2}</div>'
    + html_table_no_index(df_iqr, 'tbl-app2')
    + f'<div class="note">{note_app2}</div>'
    + '</div></body></html>'
)
IQR_EFFECTS_HTML_PATH.write_text(html_app2, encoding='utf-8')

if WRITE_LEGACY_POSITIVE_SALES_OUTPUTS:
    raise NotImplementedError(
        'Legacy positive-sales-only category outputs are disabled by default. '
        'If needed later, implement them from category_regression_positive_sales_legacy_panel '
        'and save only under outputs/regression_analysis/legacy_original_reproduction/.'
    )
else:
    print(
        'Legacy positive-sales-only category outputs are disabled. '
        'Active thesis-ready category tables use the zero-retaining open-store-day sample.'
    )

for path in [
    CATEGORY_SUMMARY_HTML_PATH,
    SIGNIFICANCE_HTML_PATH,
    IQR_EFFECTS_HTML_PATH,
]:
    print(f'Saved: {project_relative(path)}')

display(df_main)
display(appendix1_season)
display(appendix1_total)
display(df_iqr)


## Store-day appendix regression panel

The store-day appendix panel uses reconstructed MET Nordic Analysis realised weather filtered to the selected aggregation window. Weather variables are mapped to the same feature names as the previous specification to preserve formula and table comparability.


In [ ]:
def build_store_day_panel(panel):
    source = panel.loc[panel['Analyse_Kategori'].isin(ANALYSIS_CATEGORIES)]
    dim_cols = ['DatoSolgt', 'AvdelingID', 'AvdelingTekst', 'Region', 'Ukedag', 'Årstid', 'Helligdager', 'Closed']
    weather_cols = [
        'AT_percentile',
        'AT_percentile_sq',
        'Precip_pct',
        'Precip_pct_sq',
        'Cloud_pct',
        'Cloud_pct_sq',
    ]
    dim_weather = source[dim_cols + weather_cols].drop_duplicates(subset=['DatoSolgt', 'AvdelingID'])
    store_day_sales = (
        source.groupby(['DatoSolgt', 'AvdelingID'], observed=True, sort=False, as_index=False)
        .agg(
            TotalSales=('Antall_total', 'sum'),
            TotalCampaign=('Antall_campaign', 'sum'),
            TotalOrdinary=('Antall_ordinary', 'sum'),
        )
    )
    store_day = store_day_sales.merge(dim_weather, on=['DatoSolgt', 'AvdelingID'], how='left')
    store_day['CampaignDay_total'] = (store_day['TotalCampaign'] > 0).astype('int8')
    store_day['log_TotalSales'] = np.log1p(store_day['TotalSales'])
    store_day['StoreID'] = store_day['AvdelingID'].astype(str)
    store_day['Weekday'] = store_day['Ukedag'].astype(str)
    store_day['Season'] = store_day['Årstid'].astype('string').replace(SEASON_MAP).astype(str)
    store_day['Helligdager'] = pd.to_numeric(store_day['Helligdager'], errors='coerce').fillna(0).astype(int)
    return store_day


store_day_panel = build_store_day_panel(regression_panel)
STORE_DAY_REQUIRED_COLUMNS = [
    'log_TotalSales',
    'TotalSales',
    'Weekday',
    'Season',
    'StoreID',
    'Helligdager',
    'CampaignDay_total',
    'AT_percentile',
    'AT_percentile_sq',
    'Precip_pct',
    'Precip_pct_sq',
    'Cloud_pct',
    'Cloud_pct_sq',
]

store_day_regression_panel = store_day_panel.loc[
    store_day_panel['Closed'].eq(0) & store_day_panel['TotalSales'].gt(0)
].copy()
for column in [
    'TotalSales',
    'CampaignDay_total',
    'AT_percentile',
    'AT_percentile_sq',
    'Precip_pct',
    'Precip_pct_sq',
    'Cloud_pct',
    'Cloud_pct_sq',
]:
    store_day_regression_panel[column] = pd.to_numeric(store_day_regression_panel[column], errors='coerce')
store_day_regression_panel = (
    store_day_regression_panel.replace([np.inf, -np.inf], np.nan)
    .dropna(subset=STORE_DAY_REQUIRED_COLUMNS)
)

report_memory(store_day_regression_panel, 'store_day_regression_panel')
print(f'Unique store-days: {store_day_regression_panel[["DatoSolgt", "AvdelingID"]].drop_duplicates().shape[0]:,}')


## Store-day appendix regressions and VIF diagnostics

This section recreates the store-day appendix ladder and VIF diagnostics for the linear benchmark models.


In [ ]:
store_campaign_ctrl = 'CampaignDay_total'
store_base_controls = f'C(Weekday) + C(Season) + C(StoreID) + Helligdager + {store_campaign_ctrl}'
store_base_controls_complete = f'C(Weekday) + C(StoreID) + Helligdager + {store_campaign_ctrl}'
store_season_term = SEASON_TERM

store_formula_control = f'log_TotalSales ~ {store_base_controls}'
store_formula_weather = f'log_TotalSales ~ AT_percentile + Precip_pct + Cloud_pct + {store_base_controls}'
store_formula_weather_nl = (
    'log_TotalSales ~ AT_percentile + AT_percentile_sq + '
    'Precip_pct + Precip_pct_sq + '
    f'Cloud_pct + Cloud_pct_sq + {store_base_controls}'
)
store_formula_weather_seasonal = (
    f'log_TotalSales ~ {store_season_term}*(AT_percentile + Precip_pct + Cloud_pct) + '
    f'{store_base_controls_complete}'
)
store_formula_weather_complete = (
    f'log_TotalSales ~ {store_season_term}*(AT_percentile + AT_percentile_sq + '
    f'Precip_pct + Precip_pct_sq + Cloud_pct + Cloud_pct_sq) + {store_base_controls_complete}'
)

STORE_DAY_MODEL_SPECS = [
    ('(i) Control', store_formula_control),
    ('(ii) Weather', store_formula_weather),
    ('(iii) Weather (non-linear)', store_formula_weather_nl),
    ('(iv) Weather (seasonal)', store_formula_weather_seasonal),
    ('(v) Weather (complete)', store_formula_weather_complete),
]


def fmt_b_t_p(coef, t_value, p_value, digits=4):
    return f'{coef:.{digits}f}{star(p_value)} ({t_value:.2f})'


def seasonal_slope(result, base_var, season, season_ref='Spring', season_term=store_season_term, digits=4):
    main = base_var
    if main not in result.params.index:
        return ''
    coef = float(result.params[main])
    cov = result.cov_params()
    if season != season_ref:
        interaction = f'{season_term}[T.{season}]:{base_var}'
        if interaction not in result.params.index:
            return ''
        coef = coef + float(result.params[interaction])
        variance = float(cov.loc[main, main] + cov.loc[interaction, interaction] + 2 * cov.loc[main, interaction])
    else:
        variance = float(cov.loc[main, main])
    se = np.sqrt(variance) if variance >= 0 else np.nan
    t_value = coef / se if np.isfinite(se) and se > 0 else np.nan
    p_value = 2 * stats.t.sf(np.abs(t_value), df=result.df_resid) if np.isfinite(t_value) else np.nan
    return fmt_b_t_p(coef, t_value, p_value, digits=digits)


def get_terms_containing(result, substrings):
    params = list(result.params.index)
    return [parameter for parameter in params if any(substring in parameter for substring in substrings)]


def ftest_terms(result, terms):
    terms = [term for term in terms if term in result.params.index]
    if len(terms) == 0:
        return np.nan, np.nan
    param_names = list(result.params.index)
    restriction = np.zeros((len(terms), len(param_names)))
    for row_index, term in enumerate(terms):
        restriction[row_index, param_names.index(term)] = 1.0
    test = result.f_test(restriction)
    f_value = float(np.asarray(test.fvalue).squeeze())
    p_value = float(np.asarray(test.pvalue).squeeze())
    return f_value, p_value


def fmt_f_p(f_value, p_value, digits_f=2, digits_p=3):
    if not np.isfinite(f_value) or not np.isfinite(p_value):
        return ''
    p_text = '<0.001' if p_value < 0.001 else f'{p_value:.{digits_p}f}'
    return f'{f_value:.{digits_f}f} [p={p_text}]'


store_fits_plain, store_fits_cluster = fit_model_ladder(store_day_regression_panel, STORE_DAY_MODEL_SPECS)
store_cols = [name for name, _ in STORE_DAY_MODEL_SPECS]

ftest_map = {
    '(ii) Weather': {
        'joint_terms': ['AT_percentile', 'Precip_pct', 'Cloud_pct'],
        'added_terms': ['AT_percentile', 'Precip_pct', 'Cloud_pct'],
    },
    '(iii) Weather (non-linear)': {
        'joint_terms': [
            'AT_percentile', 'AT_percentile_sq',
            'Precip_pct', 'Precip_pct_sq',
            'Cloud_pct', 'Cloud_pct_sq',
        ],
        'added_terms': ['AT_percentile_sq', 'Precip_pct_sq', 'Cloud_pct_sq'],
    },
}
res_iv = store_fits_cluster['(iv) Weather (seasonal)']
joint_iv = get_terms_containing(res_iv, ['AT_percentile', 'Precip_pct', 'Cloud_pct'])
ftest_map['(iv) Weather (seasonal)'] = {'joint_terms': joint_iv, 'added_terms': joint_iv}
res_v = store_fits_cluster['(v) Weather (complete)']
joint_v = get_terms_containing(
    res_v,
    ['AT_percentile', 'AT_percentile_sq', 'Precip_pct', 'Precip_pct_sq', 'Cloud_pct', 'Cloud_pct_sq'],
)
added_v = get_terms_containing(res_v, ['AT_percentile_sq', 'Precip_pct_sq', 'Cloud_pct_sq'])
ftest_map['(v) Weather (complete)'] = {'joint_terms': joint_v, 'added_terms': added_v}

store_seasons = ['Spring', 'Summer', 'Fall', 'Winter']
weather_blocks = [
    ('Apparent Temp', 'AT_percentile', 'AT_percentile_sq'),
    ('Cloud', 'Cloud_pct', 'Cloud_pct_sq'),
    ('Precipitation', 'Precip_pct', 'Precip_pct_sq'),
]
rows_global = [('Constant', 'Intercept')]
rows_seasonal_linear = []
rows_seasonal_quad = []
for label, base, base_square in weather_blocks:
    for season in store_seasons:
        rows_seasonal_linear.append((f'{label}\n{season}', (base, season)))
        rows_seasonal_quad.append((f'({label}\n{season})²', (base_square, season)))
rows_global += [
    ('Apparent Temp', 'AT_percentile'),
    ('Apparent Temp²', 'AT_percentile_sq'),
    ('Cloud', 'Cloud_pct'),
    ('Cloud²', 'Cloud_pct_sq'),
    ('Precipitation', 'Precip_pct'),
    ('Precipitation²', 'Precip_pct_sq'),
]
store_panelA = pd.DataFrame(
    '',
    index=(
        [row[0] for row in rows_global]
        + [row[0] for row in rows_seasonal_linear]
        + [row[0] for row in rows_seasonal_quad]
    ),
    columns=store_cols,
)

for column in store_cols:
    result = store_fits_cluster[column]
    for label, variable in rows_global:
        store_panelA.loc[label, column] = format_regression_cell(result, variable)
    if column in ['(iv) Weather (seasonal)', '(v) Weather (complete)']:
        for label, (variable, season) in rows_seasonal_linear:
            store_panelA.loc[label, column] = seasonal_slope(
                result,
                variable,
                season,
                season_ref='Spring',
                season_term=store_season_term,
            )
    if column == '(v) Weather (complete)':
        for label, (variable, season) in rows_seasonal_quad:
            store_panelA.loc[label, column] = seasonal_slope(
                result,
                variable,
                season,
                season_ref='Spring',
                season_term=store_season_term,
            )

global_labels_no_const = [label for label, _ in rows_global if label != 'Constant']
for column in ['(iv) Weather (seasonal)', '(v) Weather (complete)']:
    if column in store_panelA.columns:
        store_panelA.loc[global_labels_no_const, column] = ''

store_panelB = pd.DataFrame(
    '',
    index=[
        'Controls',
        'Observations (#Stores)',
        'AIC/BIC',
        'MAPE',
        'Adjusted R²',
        'Δ Adjusted R² (pp)',
        'F-test: weather block',
        'F-test: model-specific added weather terms',
    ],
    columns=store_cols,
)
adj0 = store_fits_plain['(i) Control'].rsquared_adj
observations = int(store_fits_plain['(i) Control'].nobs)
n_stores = int(store_day_regression_panel['StoreID'].nunique())

for column in store_cols:
    model = store_fits_plain[column]
    store_panelB.loc['Controls', column] = 'Yes'
    store_panelB.loc['Observations (#Stores)', column] = f'{observations:,} ({n_stores})'
    store_panelB.loc['AIC/BIC', column] = f'{model.aic:,.0f}/{model.bic:,.0f}'
    store_panelB.loc['MAPE', column] = (
        f'{positive_sales_mape(model, store_day_regression_panel, outcome_column="TotalSales"):.1f}%'
    )
    store_panelB.loc['Adjusted R²', column] = f'{100 * model.rsquared_adj:.1f}%'
    store_panelB.loc['Δ Adjusted R² (pp)', column] = (
        'n.a.'
        if column == '(i) Control'
        else f'{100 * (model.rsquared_adj - adj0):.1f}'
    )
    if column in ftest_map:
        clustered_result = store_fits_cluster[column]
        f_joint, p_joint = ftest_terms(clustered_result, ftest_map[column]['joint_terms'])
        f_added, p_added = ftest_terms(clustered_result, ftest_map[column]['added_terms'])
        store_panelB.loc['F-test: weather block', column] = fmt_f_p(f_joint, p_joint)
        store_panelB.loc['F-test: model-specific added weather terms', column] = fmt_f_p(f_added, p_added)
    else:
        store_panelB.loc['F-test: weather block', column] = 'n.a.'
        store_panelB.loc['F-test: model-specific added weather terms', column] = 'n.a.'

store_panelA.to_csv(STORE_DAY_PANELA_CSV_PATH)
store_panelB.to_csv(STORE_DAY_PANELB_CSV_PATH)


def df_to_hovedfilen_index_html(frame):
    table = frame.copy()
    table.index = [str(index).replace('\n', '<br>') for index in table.index]
    return table.to_html(escape=False)


store_title = (
    'TABLE 1 - Total daily sales (store-day), reconstructed realised weather '
    f'({SELECTED_REALISED_WEATHER_WINDOW})'
)
store_note = (
    f'Notes: Sample includes store-days when the location is open (TotalSales > 0). '
    f'Realised weather is reconstructed MET Nordic Analysis weather filtered to {SELECTED_REALISED_WEATHER_WINDOW} '
    f'and mapped into the established weather feature names before percentile construction. '
    f't-statistics in parentheses; standard errors clustered by store. '
    f'F-tests report cluster-robust Wald tests of joint significance for weather terms. '
    f'*, **, *** denote significance at the 10%, 5%, and 1% levels.'
)
store_css = r'''
<style>
  @page { size: A4 landscape; margin: 12mm; }
  html, body { width: 297mm; margin: 0; padding: 0; }

  body { font-family: "Times New Roman", Times, serif; font-size: 9.5pt; }

  .title { text-align: center; font-size: 12pt; font-weight: bold; margin: 0 0 5px 0; }
  .rule  { border-top: 1px solid #000; margin: 6px 0 0 0; }
  .note  { margin-top: 5px; font-size: 9.5pt; text-align: left; }

  table {
    border-collapse: collapse;
    width: 100%;
    table-layout: fixed;
    border-top: 1.2px solid #000;
    border-bottom: 1.2px solid #000;
    margin: 0;
  }

  th, td { border: none; padding: 1px 3px; vertical-align: top; }
  thead th { text-align: center; font-weight: bold; padding-bottom: 3px; line-height: 1.05; }
  thead tr:last-child th { border-bottom: 1px solid #000; }

  th:first-child, td:first-child { width: 20%; text-align: left; white-space: normal; }

  th:nth-child(2), td:nth-child(2) { width: 14%; text-align: center; white-space: nowrap; }
  th:nth-child(3), td:nth-child(3) { width: 14%; text-align: center; white-space: nowrap; }
  th:nth-child(4), td:nth-child(4) { width: 17%; text-align: center; white-space: nowrap; }
  th:nth-child(5), td:nth-child(5) { width: 17%; text-align: center; white-space: nowrap; }
  th:nth-child(6), td:nth-child(6) { width: 18%; text-align: center; white-space: nowrap; }

  @media print {
    thead { display: table-row-group; }
    tfoot { display: table-row-group; }
  }
</style>
'''
store_html = (
    '<html><head>' + store_css + '</head><body>'
    + '<div class="page">'
    + f'<div class="title">{store_title}</div>'
    + df_to_hovedfilen_index_html(store_panelA)
    + '<div class="rule"></div>'
    + df_to_hovedfilen_index_html(store_panelB)
    + f'<div class="note">{store_note}</div>'
    + '</div></body></html>'
)
STORE_DAY_BOOKTABS_HTML_PATH.write_text(store_html, encoding='utf-8')


def compute_vif(formula, data, drop_intercept=True):
    _y, design = dmatrices(formula, data=data, return_type='dataframe')
    if drop_intercept and 'Intercept' in design.columns:
        design = design.drop(columns='Intercept')
    x = np.asarray(design.values, dtype='float64', order='C')
    rows = []
    for idx, variable in enumerate(design.columns):
        y_var = x[:, idx]
        x_other = np.delete(x, idx, axis=1)
        x_other = np.column_stack([np.ones(x_other.shape[0]), x_other])
        xtx = np.einsum('ni,nj->ij', x_other, x_other, optimize=False)
        xty = np.einsum('ni,n->i', x_other, y_var, optimize=False)
        try:
            beta = np.einsum('ij,j->i', solve_gauss_jordan(xtx, np.eye(xtx.shape[0])), xty, optimize=False)
            fitted = np.einsum('ni,i->n', x_other, beta, optimize=False)
            resid = y_var - fitted
            tss = float(np.sum((y_var - y_var.mean()) ** 2))
            rss = float(np.sum(resid**2))
            r2 = 1 - rss / tss if tss > 0 else np.nan
            vif_value = 1 / max(1 - r2, np.finfo(float).eps) if np.isfinite(r2) else np.nan
        except Exception:
            vif_value = np.nan
        rows.append({'Variable': variable, 'VIF': vif_value})
    vif = pd.DataFrame(rows)
    return vif.sort_values('VIF', ascending=False).reset_index(drop=True)


vif_control = compute_vif(store_formula_control, store_day_regression_panel)
vif_weather = compute_vif(store_formula_weather, store_day_regression_panel)
vif_control.to_csv(VIF_CONTROL_CSV_PATH, index=False)
vif_weather.to_csv(VIF_WEATHER_CSV_PATH, index=False)

mean_vif_control = vif_control['VIF'].mean()
max_vif_control = vif_control['VIF'].max()
max_var_control = vif_control.loc[vif_control['VIF'].idxmax(), 'Variable']
mean_vif_weather = vif_weather['VIF'].mean()
max_vif_weather = vif_weather['VIF'].max()
max_var_weather = vif_weather.loc[vif_weather['VIF'].idxmax(), 'Variable']
corr_precip_cloud = store_day_regression_panel[['Precip_pct', 'Cloud_pct']].corr().loc['Precip_pct', 'Cloud_pct']

label_map = {
    'CampaignDay_total': 'Campaign day',
    'Precip_pct': 'Precipitation',
    'Cloud_pct': 'Cloud cover',
    'AT_percentile': 'Apparent temperature',
    'Helligdager': 'Holiday',
}
max_var_control_clean = label_map.get(max_var_control, max_var_control)
max_var_weather_clean = label_map.get(max_var_weather, max_var_weather)

vif_panelA = pd.DataFrame(
    [
        {
            'Model': '(i) Control',
            'Mean VIF': f'{mean_vif_control:.2f}',
            'Max VIF': f'{max_vif_control:.2f}',
            'Variable with max VIF': max_var_control_clean,
        },
        {
            'Model': '(ii) Weather',
            'Mean VIF': f'{mean_vif_weather:.2f}',
            'Max VIF': f'{max_vif_weather:.2f}',
            'Variable with max VIF': max_var_weather_clean,
        },
    ]
)
vif_precip = vif_weather.loc[vif_weather['Variable'] == 'Precip_pct', 'VIF'].iloc[0]
vif_cloud = vif_weather.loc[vif_weather['Variable'] == 'Cloud_pct', 'VIF'].iloc[0]
vif_at = vif_weather.loc[vif_weather['Variable'] == 'AT_percentile', 'VIF'].iloc[0]
vif_panelB = pd.DataFrame(
    [
        {'Statistic': 'VIF, Apparent temperature', 'Value': f'{vif_at:.2f}'},
        {'Statistic': 'VIF, Precipitation', 'Value': f'{vif_precip:.2f}'},
        {'Statistic': 'VIF, Cloud cover', 'Value': f'{vif_cloud:.2f}'},
        {'Statistic': 'Correlation, Precipitation vs Cloud cover', 'Value': f'{corr_precip_cloud:.2f}'},
    ]
)
vif_panelA.to_csv(VIF_PANELA_CSV_PATH, index=False)
vif_panelB.to_csv(VIF_PANELB_CSV_PATH, index=False)

vif_title = 'TABLE A.1 - Multicollinearity diagnostics for linear benchmark models'
vif_note = (
    'Notes: VIFs are computed for the linear benchmark specifications only, excluding '
    'quadratic and interaction terms. The weather model shows a higher maximum VIF, '
    'driven mainly by precipitation and cloud cover. Their positive pairwise correlation '
    'indicates moderate overlap between related weather dimensions, but not severe '
    'multicollinearity in the benchmark analysis.'
)
vif_css = r'''
<style>
  @page { size: A4 portrait; margin: 16mm; }
  body { font-family: "Times New Roman", Times, serif; font-size: 10pt; margin: 0; padding: 0; }
  .title { text-align: center; font-size: 12pt; font-weight: bold; margin-bottom: 8px; }
  .rule  { border-top: 1px solid #000; margin: 6px 0; }
  .note  { margin-top: 8px; font-size: 9.5pt; text-align: left; }
  table {
    border-collapse: collapse;
    width: 100%;
    border-top: 1.2px solid #000;
    border-bottom: 1.2px solid #000;
    margin-bottom: 8px;
  }
  th, td {
    border: none;
    padding: 4px 6px;
    vertical-align: top;
  }
  thead th {
    text-align: center;
    font-weight: bold;
    border-bottom: 1px solid #000;
  }
  tbody td:first-child {
    text-align: left;
  }
  tbody td:not(:first-child) {
    text-align: center;
  }
</style>
'''
vif_html = (
    '<html><head>' + vif_css + '</head><body>'
    + f'<div class="title">{vif_title}</div>'
    + vif_panelA.to_html(index=False, escape=False)
    + '<div class="rule"></div>'
    + vif_panelB.to_html(index=False, escape=False)
    + f'<div class="note">{vif_note}</div>'
    + '</body></html>'
)
VIF_SUMMARY_HTML_PATH.write_text(vif_html, encoding='utf-8')

for path in [STORE_DAY_BOOKTABS_HTML_PATH, VIF_SUMMARY_HTML_PATH]:
    print(f'Saved: {project_relative(path)}')
display(store_panelA)
display(store_panelB)
display(vif_panelA)
display(vif_panelB)


## Quadratic weather-effect figures

These optional figures visualize the complete category models by season using the quadratic-weather plotting approach from the source notebook.


In [ ]:
def quad_effect_pct(model, linear_var, square_var, season, xgrid, x0=0.5, baseline='Spring'):
    b1 = season_coef(model, linear_var, season, baseline)
    b2 = season_coef(model, square_var, season, baseline)
    dlog = b1 * (xgrid - x0) + b2 * (xgrid**2 - x0**2)
    return 100 * (np.exp(dlog) - 1)


if RENDER_REGRESSION_FIGURES:
    xgrid = np.linspace(0, 1, 101)
    for category, model in complete_models_cluster.items():
        fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
        fig.suptitle(f'Quadratic realised-weather analysis: {category}', fontsize=14)
        for axis, season in zip(axes, SEASONS):
            axis.plot(
                xgrid * 100,
                quad_effect_pct(model, 'AT_percentile', 'AT_percentile_sq', season, xgrid),
                label='AT',
            )
            axis.plot(
                xgrid * 100,
                quad_effect_pct(model, 'Precip_pct', 'Precip_pct_sq', season, xgrid),
                '--',
                label='Precip',
            )
            axis.plot(
                xgrid * 100,
                quad_effect_pct(model, 'Cloud_pct', 'Cloud_pct_sq', season, xgrid),
                ':',
                label='Cloud',
            )
            axis.axhline(0, color='grey', linewidth=1)
            axis.set_title(season)
            axis.set_xlabel('Percentile (0-100)')
            axis.grid(alpha=0.2)
        axes[0].set_ylabel('Change in sales (%) relative to percentile 50')
        axes[0].legend()
        plt.tight_layout()
        safe_name = re.sub(r'[^A-Za-z0-9]+', '_', category).strip('_').lower()
        figure_path = OUTPUT_FIGURE_DIR / f'quadratic_weather_effects_{safe_name}.png'
        plt.savefig(figure_path, dpi=200)
        plt.close()
        print(f'Saved: {project_relative(figure_path)}')
else:
    print('Quadratic weather-effect figures skipped (RENDER_REGRESSION_FIGURES=False).')


## Output summary

The final cell lists expected output files and confirms that regression outputs were written under the project output structure.


In [ ]:
output_paths = [
    CHECKS_PATH,
    ORIGINAL_DESCRIPTIVE_CSV_PATH,
    ORIGINAL_DESCRIPTIVE_HTML_PATH,
    ORIGINAL_DESCRIPTIVE_MISSING_CSV_PATH,
    ORIGINAL_DESCRIPTIVE_MISSING_HTML_PATH,
    DESCRIPTIVE_CSV_PATH,
    DESCRIPTIVE_HTML_PATH,
    ZERO_RATE_PATH,
    WEATHER_MISSINGNESS_PATH,
    DEPENDENT_SUMMARY_PATH,
    WEATHER_CORR_CSV_PATH,
    WEATHER_WINDOW_DECISION_CSV_PATH,
    WEATHER_WINDOW_DECISION_HTML_PATH,
    CATEGORY_LADDER_CSV_PATH,
    CATEGORY_LADDER_HTML_PATH,
    STORE_DAY_PANELA_CSV_PATH,
    STORE_DAY_PANELB_CSV_PATH,
    STORE_DAY_BOOKTABS_HTML_PATH,
    VIF_CONTROL_CSV_PATH,
    VIF_WEATHER_CSV_PATH,
    VIF_PANELA_CSV_PATH,
    VIF_PANELB_CSV_PATH,
    VIF_SUMMARY_HTML_PATH,
    CATEGORY_SUMMARY_CSV_PATH,
    CATEGORY_SUMMARY_HTML_PATH,
    SIGNIFICANCE_PANELA_CSV_PATH,
    SIGNIFICANCE_PANELB_CSV_PATH,
    SIGNIFICANCE_HTML_PATH,
    IQR_EFFECTS_CSV_PATH,
    IQR_EFFECTS_HTML_PATH,
]

output_status = pd.DataFrame(
    {
        'output': [project_relative(path) for path in output_paths],
        'exists': [path.exists() for path in output_paths],
    }
)
display(output_status)

gc.collect()
print(
    'Regression analysis notebook complete. Active regressions use reconstructed '
    f'realised weather window: {SELECTED_REALISED_WEATHER_WINDOW}. HTML tables are '
    'saved under outputs/regression_analysis/html when the notebook is executed.'
)
